# Rebuild the ReDraw split of the thesis (ui2compose, SoftwareX revision)

This notebook rebuilds the train/val/test split from the thesis and the paper, checks it, and puts everything the laptop needs into **one zip file on your Google Drive**. Once you have allowed Google Drive access (in the first minute), it runs by itself.

**What it does**

1. Downloads the ReDraw dataset (`ReDraw-Final-Google-Play-Dataset.tar.gz`, 11.6 GB) from Zenodo record 2530277, checks its MD5 against the value Zenodo publishes, unpacks it and deletes the download.
2. Rebuilds the split with the **unchanged preprocessing code of ui2compose v1.1.0** (`parse_redraw` followed by `build_dataset`), with the commands documented in the v1.1.0 README.
3. **Checks the split** against the thesis: 14,955 / 1,869 / 1,870 screens, 172,732 training instances, the per-class counts of Table 3.1, all 10 bundled `examples/*.png` in test, and 214 test screens of apps that never occur in training (1,392 of 1,595 test apps also in train). If any of these differ, it retries with the older tags v1.0.2 and v1.0.1, then tests a few diagnostic variants of the same code.
4. Re-evaluates the released weights (`runs/oversample_5k/weights/best.pt`) on val and test and compares the result with the published values (test mAP@50 48.70 %, val mAP@50 51.82 %; re-evaluation in Table A.2: 48.74 % / 51.87 %).
5. Hashes every screen (MD5 + dHash) with the new `app_split` tool from v1.2.0, which is embedded here because it is not on GitHub yet. It writes the manifest and the unseen-app lists.
6. Zips the val+test images, the labels of all splits, the raw XML view hierarchies of the val+test screens, the manifest, the metrics and the provenance information (about 2.2–2.6 GB), and copies the zip to `MyDrive/ui2compose_revision/` together with `REPORT.txt`.

**How to start it**

1. Open [colab.research.google.com](https://colab.research.google.com), choose *File → Upload notebook* and select this file (`redraw_rebuild_colab.ipynb`).
2. Keep the default **CPU** runtime. A GPU is not needed and only makes the metric check ~40 min faster; free GPU time is limited, and if Colab takes the machine away during the long download, everything starts again.
3. Choose *Runtime → Run all*. Colab warns that the notebook was not written by Google: choose **Run anyway**.
4. **Within the first minute**, a pop-up asks for access to Google Drive: choose your account and allow access. After that, nothing needs your attention.
5. Keep the browser tab open and the computer awake (no sleep mode). **Look at the tab every 30–60 minutes**: free Colab sometimes asks "Are you still there?"; click the button, otherwise the session is stopped.

**Expected runtime:** about **2.5–4.5 hours** in total on a CPU runtime:
* download: 1–3 h (the slow part; the speed from Zenodo was 1.4–2.3 MB/s when measured, Colab may be faster; speed and ETA are printed every minute);
* unpacking and preprocessing: ~20–30 min;
* metric check: ~40–55 min on CPU (~10 min on a T4 GPU);
* hashing (app_split): ~15 min;
* zip and copy to Drive: ~10–15 min.

If the split check fails, the fallback runs add up to ~1 h. The machine needs about 40 GB of free disk (free Colab provides it) and your Google Drive needs about 3.5 GB of free space; the notebook checks both at the start.

**If the connection drops or the runtime restarts:** choose *Run all* again. Finished steps are skipped (markers in `/content/work/markers`) and the download resumes where it stopped. If Colab gives you a **new machine**, `/content` is empty and everything starts from the beginning.

**What to send back:** `REPORT.txt` from `MyDrive/ui2compose_revision/`. It is updated after every step, so it is useful even if the run stops halfway. It starts with an OVERALL list of PASS/FAIL and a **KEY NUMBERS** block. If something says FAIL, also send `ui2compose_diagnostics.zip` from the same folder (5–20 MB, no images). Download the big zip `ui2compose_redraw_valtest_<tag>.zip` to the laptop; Drive warns that it cannot scan such a large file for viruses: choose **Download anyway**.

**If you see FAIL:** do not change anything. When a split check fails, the notebook stops before building the zip; `REPORT.txt` and `ui2compose_diagnostics.zip` are already on Drive, so send them back. To build the zip anyway, tick `FORCE` in Step 1 and choose *Run all* again; finished steps are skipped.

## Step 1 – Settings

You normally do **not** need to change anything here. The two check boxes are:
* `DRY_RUN`: for developers only. It runs a local test on a tiny synthetic archive. Leave it unticked.
* `FORCE`: tick it only if Step 5 reported FAIL and you still want the zip. Then choose *Run all* again.

The thesis reference values (split sizes, Table 3.1 per-class counts, Table A.2) are in the next cell and must not be edited.

In [ ]:
#@title Step 1 - Settings (the only cell you might edit)
DRY_RUN = False  #@param {type:"boolean"}
FORCE = False  #@param {type:"boolean"}

WORK = '/content/work'                                        # scratch space on the Colab machine (~40 GB)
OUT_DRIVE_DIR = '/content/drive/MyDrive/ui2compose_revision'  # the zip and REPORT.txt are copied here
REPO_URL = 'https://github.com/gbilgeturk/ui2compose'
REPO_TAG = 'v1.1.0'                    # the code that produced the thesis split
FALLBACK_TAGS = ['v1.0.2', 'v1.0.1']   # tried automatically if REPO_TAG does not reproduce the split

ZENODO_RECORD = '2530277'                                # "The ReDraw Dataset" (Moran et al.)
ZENODO_FILE = 'ReDraw-Final-Google-Play-Dataset.tar.gz'  # 11.58 GB: full screenshots + uiautomator XML
TARBALL_DIR = None    # None: download into WORK and delete the archive after unpacking. A Drive folder
                      # (e.g. '/content/drive/MyDrive/redraw_cache') keeps the archive across Colab
                      # machines -- only if your Drive has 12 GB free.
DELETE_TARBALL_AFTER_EXTRACT = True   # (a TARBALL_DIR copy is never deleted)
USE_SYMLINKS = True   # build_dataset --symlink: identical split and labels, saves ~11 GB of disk
MIN_FREE_GB = 40      # local disk needed on the Colab machine
DRIVE_MIN_FREE_GB = 3.5   # Google Drive space needed for the zip (~2.5 GB), REPORT.txt and the diagnostics

ULTRALYTICS_VERSION = '8.4.24'        # version the released weights were trained with
# YOLO.val() settings: Ultralytics validation defaults; batch 16 = the training-time validation batch (8 x 2)
VAL_SETTINGS = dict(imgsz=640, batch=16, conf=0.001, iou=0.7, max_det=300, rect=True)
ALSO_RUN_FP16_ON_GPU = True           # the training-time validation ran in FP16 (AMP); also evaluate that way on a GPU
MAP_TOL = 0.005                       # allowed |difference| of mAP@50 and mAP@50-95 from results/*.json (0.5 points)
CLASS_TOL = 0.02                      # allowed |difference| of every per-class AP@50 (test)
REEVAL_TOL = 0.0015                   # allowed |difference| of mAP@50 from the thesis re-evaluation (Table A.2; 0.15 points)

TRY_HYPOTHESES = True                 # diagnostic variants, only if no tag reproduces the split (see Step 5)
EXPECT_FINGERPRINTS_JSON = None       # optional: reference summary.json for app_split --expect-fingerprints
VERIFY_DRIVE_COPY = True              # read the zip back from Drive and compare its SHA-256

import os
DRY_RUN_RECORD_JSON = DRY_RUN_ARCHIVE = None
if DRY_RUN:  # local test: synthetic archive instead of Zenodo, no Google Drive, all output written locally
    _dry = os.environ.get('UI2C_DRYRUN_DIR', os.path.abspath('ui2c_dryrun'))
    WORK = os.path.join(_dry, 'work')
    OUT_DRIVE_DIR = os.path.join(_dry, 'drive_out')
    REPO_URL = os.environ.get('UI2C_DRYRUN_REPO_URL', REPO_URL)
    DRY_RUN_RECORD_JSON = os.environ.get('UI2C_DRYRUN_RECORD_JSON')  # Zenodo-API look-alike for the synthetic archive
    DRY_RUN_ARCHIVE = os.environ.get('UI2C_DRYRUN_ARCHIVE')          # or just the synthetic archive itself
    MIN_FREE_GB = 1
print(f'DRY_RUN={DRY_RUN}  FORCE={FORCE}\nWORK={WORK}\nOUT_DRIVE_DIR={OUT_DRIVE_DIR}\n'
      f'REPO={REPO_URL} @ {REPO_TAG} (fallbacks {FALLBACK_TAGS})')

In [ ]:
#@title Step 1b - Reference values from the thesis and the repository (do not edit)
EXPECTED = {
    # Section 3.2.3 / Table A.2: the split
    'train_screens': 14955, 'val_screens': 1869, 'test_screens': 1870, 'total_pairs': 18694,
    'train_instances': 172732,           # Table 3.1 Train(O): label lines in labels/train (before oversampling)
    'examples_in_test': 10,              # examples/*.png of the repository are test screens
    # Section 4.5.5 (Limitations): pure functions of the split (package = part of the file name before '-screens')
    'unseen_test_package_only': 214,     # test screens whose package never occurs in train
    'test_apps': 1595, 'test_apps_in_train': 1392,
    # Table A.2: images pixel-identical to a training image (white-label clones)
    'val_pixel_identical_to_train': 9, 'test_pixel_identical_to_train': 8,
    # Table A.2: re-evaluated mAP@50 of the released weights outside the training environment
    'reeval_val_mAP50': 0.5187, 'reeval_test_mAP50': 0.4874,
    'app_folders_in_archive': 6526,      # counted in a Range-request probe of the archive (informational)
}
# Table 3.1 (Section 3.2.1): instances per class in Train(O) / Validation / Test. Totals 172,732 / 21,613 / 23,046.
EXPECTED_CLASS_COUNTS = {
    'train': {'Button': 21719, 'CheckBox': 1434, 'EditText': 4883, 'ImageView': 48507, 'ListView': 3744,
              'ProgressBar': 451, 'RadioButton': 1061, 'RecyclerView': 1269, 'SeekBar': 353, 'Spinner': 626,
              'Switch': 281, 'TextView': 87463, 'WebView': 941},
    'val': {'Button': 2701, 'CheckBox': 148, 'EditText': 619, 'ImageView': 6075, 'ListView': 477,
            'ProgressBar': 56, 'RadioButton': 157, 'RecyclerView': 142, 'SeekBar': 34, 'Spinner': 76,
            'Switch': 60, 'TextView': 10954, 'WebView': 114},
    'test': {'Button': 2791, 'CheckBox': 212, 'EditText': 634, 'ImageView': 6441, 'ListView': 450,
             'ProgressBar': 50, 'RadioButton': 134, 'RecyclerView': 165, 'SeekBar': 67, 'Spinner': 68,
             'Switch': 61, 'TextView': 11865, 'WebView': 108},
}
# The published metrics are read from the cloned repository in Step 6:
#   test: results/perclass_ap_results.json['oversample_5k']  (mAP@50 0.48699, mAP@50-95 0.39109, per-class AP@50)
#   val:  results/oversample_5k_metrics.json                  (mAP@50 0.51823, mAP@50-95 0.40173; training-time value)

# What the code looked like when this notebook was written (git commits and SHA-256 of the files that
# define the split), and the SHA-256 of the embedded app_split.py (Step 7).
PINNED = {'commits': {'v1.1.0': '865bc2e5b110fd3d3f60a8fc62bdeca772f4d616',
             'v1.0.2': '0e557ab4b03169aee2f8b1be8d737d2443b096d6',
             'v1.0.1': '215fda09a348b4222efccaaf44dc4da5b95e7049'},
 'files': {'v1.1.0': {'src/ui2compose/preprocess/parse_redraw.py': '97e5b1644010095fe3d115ef5939b3e0cf12d1e7a34a2da21548232a10170e16',
                      'src/ui2compose/preprocess/build_dataset.py': '49f3d54849cdacb1a431e3b59ccd7647f908b921094dcc9932f996898ec32409',
                      'configs/dataset.yaml': '134ed8a6fcc304d7b8fa3922fb1277a2df8596bd3348b33286302b9ee0fc2d6a',
                      'runs/oversample_5k/weights/best.pt': 'b7b2987fb89d65fa5e9efd74135b4d38ebe080ec4b7a7109fab8624d2067e2ae',
                      'results/perclass_ap_results.json': 'bfe0b8b9876a875f551cb3285664df774819ceef3245d50c045eae49065e548b',
                      'results/oversample_5k_metrics.json': 'd7c1d3c1a739a1c5f310c28c2a9674dd1569a6f984510a06a05add71f155c9d2'},
           'v1.0.2': {'src/preprocess/parse_redraw.py': '006e68f4178967b45d01aa2f34951581b6f8c8612a3b15ac4ad7394e22738a1f',
                      'src/preprocess/build_dataset.py': '20de8d092706294227a8cfa11780d0dad023ee70d1b35e492ba92c2660e28313',
                      'configs/dataset.yaml': '134ed8a6fcc304d7b8fa3922fb1277a2df8596bd3348b33286302b9ee0fc2d6a',
                      'runs/oversample_5k/weights/best.pt': 'b7b2987fb89d65fa5e9efd74135b4d38ebe080ec4b7a7109fab8624d2067e2ae'},
           'v1.0.1': {'src/preprocess/parse_redraw.py': '006e68f4178967b45d01aa2f34951581b6f8c8612a3b15ac4ad7394e22738a1f',
                      'src/preprocess/build_dataset.py': '20de8d092706294227a8cfa11780d0dad023ee70d1b35e492ba92c2660e28313',
                      'configs/dataset.yaml': '134ed8a6fcc304d7b8fa3922fb1277a2df8596bd3348b33286302b9ee0fc2d6a',
                      'runs/oversample_5k/weights/best.pt': 'b7b2987fb89d65fa5e9efd74135b4d38ebe080ec4b7a7109fab8624d2067e2ae'}},
 'parse_redraw_defaults': {'min_area': 5e-05, 'dedup_iou': 0.85, 'drop_container': True, 'limit': 0},
 'build_dataset_constants': {'SEED': 42, 'SPLIT': [0.8, 0.1, 0.1]},
 'app_split_sha256': '60b693d1719f5241cf2f0bb06a4e229e8c6c44770ff7987f4fcf10736f2b388b',
 'app_split_file': 'ui2compose_app_split_v1.2.0.py'}
print('reference values loaded')

In [ ]:
#@title Step 1c - Helper functions (just run it; no need to read)
import csv, datetime, hashlib, io, json, os, platform, re, shlex, shutil, signal, subprocess, sys, time, zipfile
from collections import Counter
from pathlib import Path

WORK = Path(WORK)
D = {k: WORK / k for k in ('markers', 'state', 'logs', 'download', 'raw', 'repos', 'data', 'tools',
                           'splits', 'metrics', 'out')}
for _p in D.values():
    _p.mkdir(parents=True, exist_ok=True)
IMG_EXTS = ('.png', '.jpg', '.jpeg')
SPLITS = ('train', 'val', 'test')
STEM_RE = re.compile(r'^(?P<app>.+-screens)_screenshot_(?P<n>\d+)$')   # build_dataset output names
RAW_NAME_RE = re.compile(r'^(hierarchy_\d+\.xml|screenshot_\d+\.png)$')  # the only names found in the archive probe
DIAG_ZIP = 'ui2compose_diagnostics.zip'


class PipelineStop(Exception):
    """Stops 'Run all' with a short message instead of a long traceback."""
    def _render_traceback_(self):
        bar = '=' * 78
        return [bar, 'STOPPED: ' + str(self), bar]


def now():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')


def jdump(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    os.replace(tmp, path)


def jload(path, default=None):
    try:
        return json.loads(Path(path).read_text(encoding='utf-8'))
    except FileNotFoundError:
        return default


# Markers make every step restartable: a step that finished writes markers/<name>.done (with its results).
def done(name):
    return (D['markers'] / f'{name}.done').exists()


def mark(name, info=None):
    jdump({'finished': now(), **(info or {})}, D['markers'] / f'{name}.done')


def marker(name):
    return jload(D['markers'] / f'{name}.done', {})


def save_state(name, obj):
    jdump(obj, D['state'] / f'{name}.json')


def load_state(name, default=None):
    return jload(D['state'] / f'{name}.json', default)


def human(n):
    n = float(n)
    for unit in ('B', 'KB', 'MB', 'GB', 'TB'):
        if abs(n) < 1024 or unit == 'TB':
            return f'{n:.0f} {unit}' if unit == 'B' else f'{n:.2f} {unit}'
        n /= 1024


def free_gb(path):
    return shutil.disk_usage(path).free / 1e9


def file_hash(path, algo='sha256', progress_every=0):
    h, n, t0, last = hashlib.new(algo), 0, time.time(), time.time()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8 << 20), b''):
            h.update(chunk)
            n += len(chunk)
            if progress_every and time.time() - last > progress_every:
                last = time.time()
                print(f'  {algo}: {human(n)} read ({n / max(last - t0, 1e-9) / 1e6:.0f} MB/s)', flush=True)
    return h.hexdigest()


def qcmd(cmd):
    return ' '.join(shlex.quote(str(c)) for c in cmd)


def kill_stale(substr, programs):
    """Stops processes of an earlier, interrupted run (e.g. a curl that survived a kernel restart and would
    otherwise write into the same file as the new download). programs: program-name prefixes."""
    try:
        out = subprocess.run(['ps', '-ww', '-eo', 'pid=,args='], capture_output=True, text=True, timeout=30).stdout
    except Exception:
        return []
    killed = []
    for line in out.splitlines():
        parts = line.strip().split(None, 1)
        if len(parts) != 2 or not parts[0].isdigit() or int(parts[0]) == os.getpid():
            continue
        prog = os.path.basename(parts[1].split()[0])
        if str(substr) in parts[1] and any(prog.startswith(p) for p in programs):
            try:
                os.kill(int(parts[0]), signal.SIGTERM)
                killed.append(int(parts[0]))
            except OSError:
                pass
    if killed:
        time.sleep(2)
        for pid in killed:
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
        print(f'stopped {len(killed)} leftover process(es) of an earlier run: {killed}', flush=True)
    return killed


def run(cmd, cwd=None, log=None, check=True, echo=True, quiet=False):
    """Runs a command, streams its output, appends it to logs/<log>; raises PipelineStop on failure.
    If the cell is interrupted, the command is killed too (no process keeps running in the background)."""
    cmd = [str(c) for c in cmd]
    if not quiet:
        print('$ ' + qcmd(cmd), flush=True)
    t0, lines = time.time(), []
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         errors='replace', bufsize=1)
    try:
        for line in p.stdout:
            lines.append(line)
            if echo:
                print(line, end='', flush=True)
        rc = p.wait()
    except BaseException:
        p.kill()
        p.wait()
        if log:
            with open(D['logs'] / log, 'a', encoding='utf-8') as f:
                f.write(f'\n### {now()}  cwd={cwd}\n$ {qcmd(cmd)}\n{"".join(lines)}\n### INTERRUPTED (killed)\n')
        raise
    out = ''.join(lines)
    if log:
        with open(D['logs'] / log, 'a', encoding='utf-8') as f:
            f.write(f'\n### {now()}  cwd={cwd}\n$ {qcmd(cmd)}\n{out}\n### exit {rc} after {time.time() - t0:.0f} s\n')
    if check and rc != 0:
        raise PipelineStop(f'command failed with exit code {rc}: {qcmd(cmd)[:300]}'
                           + (f'  (log: {D["logs"] / log})' if log else ''))
    return rc, out


# ---------------- PASS/FAIL tables ----------------
def fmt(v):
    if isinstance(v, bool) or v is None:
        return str(v)
    if isinstance(v, float):
        return f'{v:.5f}'
    if isinstance(v, int):
        return f'{v:,}'
    return str(v)


def check_row(check, expected, got, ok=None, status=None, note=''):
    if status is None:
        status = 'PASS' if (ok if ok is not None else expected == got) else 'FAIL'
    return {'check': check, 'expected': expected, 'got': got, 'status': status, 'note': note}


def table_lines(title, rows):
    out = [title, '-' * len(title)]
    w = max([len(str(r['check'])) for r in rows] + [8])
    for r in rows:
        line = f"{r['status']:<5} {str(r['check']):<{w}}  expected {fmt(r['expected'])}  got {fmt(r['got'])}"
        if r.get('note'):
            line += f"  [{r['note']}]"
        out.append(line)
    return out


def set_checks(section, rows, show=True):
    allc = load_state('checks', {})
    allc[section] = rows
    save_state('checks', allc)
    if show:
        print('\n' + '\n'.join(table_lines(section, rows)))
    return all(r['status'] in ('PASS', 'INFO', 'WARN') for r in rows)


def banner(text, ok=True):
    bar = ('=' if ok else '#') * 78
    print(f'\n{bar}\n{text}\n{bar}\n', flush=True)


# ---------------- Google Drive ----------------
def drive_alive():
    """True if OUT_DRIVE_DIR is on a working Drive mount (a test file can be written)."""
    if DRY_RUN:
        Path(OUT_DRIVE_DIR).mkdir(parents=True, exist_ok=True)
        return True
    try:
        if not os.path.ismount('/content/drive'):
            return False   # never write below /content/drive without a mount (a later mount would fail)
        Path(OUT_DRIVE_DIR).mkdir(parents=True, exist_ok=True)
        # overwritten, never deleted (deleted Drive files would pile up in the Drive bin)
        (Path(OUT_DRIVE_DIR) / '.ui2compose_write_test').write_text(now())
        return True
    except OSError:
        return False


def mount_drive(force=False):
    """Connects Google Drive (Colab shows a pop-up). Raises PipelineStop with instructions if that fails."""
    if DRY_RUN:
        Path(OUT_DRIVE_DIR).mkdir(parents=True, exist_ok=True)
        print(f'DRY_RUN: Google Drive is not mounted; "Drive" output goes to {OUT_DRIVE_DIR}')
        return
    if not force and drive_alive():
        print(f'Google Drive is connected; results go to {OUT_DRIVE_DIR}')
        return
    print('Connecting Google Drive: a pop-up opens. Choose your Google account and allow access '
          '(within 2 minutes).', flush=True)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=force or os.path.ismount('/content/drive'))
    except Exception as e:
        raise PipelineStop(f'Google Drive could not be connected ({type(e).__name__}: {e}). Run this cell again '
                           '(or Runtime -> Run all) and allow access in the pop-up. Finished steps are skipped.')
    if not drive_alive():
        raise PipelineStop(f'Google Drive is mounted, but {OUT_DRIVE_DIR} cannot be written. Check that your Drive '
                           'is not full, then run this cell again.')
    print(f'Google Drive connected; results go to {OUT_DRIVE_DIR}')


def drive_free_gb():
    """Free space on Google Drive in GB as the mount reports it (None if it reports nothing useful)."""
    try:
        u = shutil.disk_usage(OUT_DRIVE_DIR if DRY_RUN else '/content/drive')
        return u.free / 1e9 if u.total > 0 else None
    except OSError:
        return None


def copy_to_drive(src, name=None, verify=False):
    """Copies a file into OUT_DRIVE_DIR (.partial, then rename). Returns (dest, sha256_ok); (None, None) if
    Drive is not connected or the copy failed (a warning is printed; the local file stays)."""
    if not drive_alive():
        return None, None
    dest = Path(OUT_DRIVE_DIR) / (name or Path(src).name)
    try:
        tmp = Path(str(dest) + '.partial')
        shutil.copyfile(src, tmp)
        os.replace(tmp, dest)
        ok = (file_hash(dest) == file_hash(src)) if verify else None
    except OSError as e:
        print(f'WARNING: copying {Path(src).name} to Google Drive failed ({e}); the file stays at {src}')
        return None, None
    return dest, ok


def drive_flush():
    """Writes everything still cached to Google Drive and unmounts it (Colab only)."""
    if DRY_RUN or not os.path.ismount('/content/drive'):
        return
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        print('Google Drive flushed and unmounted: everything written so far is on your Drive.')
    except Exception as e:
        print(f'WARNING: flushing Google Drive failed ({e}); wait a few minutes before closing the tab.')


# ---------------- repository and preprocessing ----------------
def repo_dir(tag):
    return D['repos'] / f'ui2compose_{tag}'


def clone_repo(tag):
    """Clones REPO_URL at a tag and checks commit + file hashes against PINNED."""
    dest, key = repo_dir(tag), f'clone_{tag}'
    if done(key) and dest.exists():
        info = marker(key)
        print(f'[skip] {tag} already cloned at {dest} (commit {info.get("commit")})')
        return dest, info
    if dest.exists():
        shutil.rmtree(dest)
    run(['git', '-c', 'advice.detachedHead=false', 'clone', '--quiet', '--depth', '1', '--branch', tag,
         REPO_URL, dest], log='git.log')
    commit = run(['git', '-C', dest, 'rev-parse', 'HEAD'], echo=False, quiet=True)[1].strip()
    pinned_files = PINNED['files'].get(tag, {})
    files = {rel: (file_hash(dest / rel) if (dest / rel).exists() else None) for rel in pinned_files}
    problems = []
    if PINNED['commits'].get(tag) and commit != PINNED['commits'][tag]:
        problems.append(f'commit {commit} differs from the pinned {PINNED["commits"][tag]}')
    problems += [f'{rel}: SHA-256 {h} differs from the pinned value' for rel, h in files.items()
                 if h != pinned_files[rel]]
    info = {'tag': tag, 'url': REPO_URL, 'commit': commit, 'file_sha256': files, 'problems': problems}
    mark(key, info)
    print(f'{tag}: commit {commit} - ' + ('commit and file hashes match the pinned values' if not problems
                                          else 'WARNING: ' + '; '.join(problems)))
    return dest, info


def find_script(clone, name):
    for rel in (f'src/ui2compose/preprocess/{name}.py', f'src/preprocess/{name}.py'):
        if (Path(clone) / rel).exists():
            return Path(clone) / rel
    raise PipelineStop(f'{name}.py not found in {clone}')


# Imports a script by path and prints the argparse result for the given argv (the values main() will use)
_ARGS_PROBE = '''
import importlib.util, json, sys
spec = importlib.util.spec_from_file_location("probe_mod", sys.argv[1])
m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
sys.argv = [sys.argv[1]] + sys.argv[2:]
consts = {k: getattr(m, k) for k in ("SEED", "SPLIT") if hasattr(m, k)}
print("ARGS_JSON " + json.dumps({"args": vars(m.parse_args()), "constants": consts}))
'''


def effective_args(script, argv):
    _, out = run([sys.executable, '-c', _ARGS_PROBE, script, *argv], echo=False, quiet=True)
    line = [l for l in out.splitlines() if l.startswith('ARGS_JSON ')][-1]
    return json.loads(line[len('ARGS_JSON '):])


def raw_root():
    info = marker('extract')
    if not info:
        raise PipelineStop('Step 3 (download and unpack) has not finished - run Step 3 first.')
    return Path(info['raw_root'])


# Runs the UNCHANGED build_dataset.py, but with the case-insensitive path order of Windows (diagnostic only):
# its module-level 'sorted' is replaced by one that compares str(path).lower(), as PureWindowsPath does.
_CASEFOLD_BUILD = '''
import builtins, importlib.util, sys
spec = importlib.util.spec_from_file_location("build_dataset", sys.argv[1])
m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
m.sorted = lambda it, key=None, reverse=False: builtins.sorted(it, key=key or (lambda p: str(p).lower()), reverse=reverse)
sys.argv = [sys.argv[1]] + sys.argv[2:]
m.main()
'''


def write_yaml13(yolo, names):
    (Path(yolo) / 'dataset.yaml').write_text(
        f'path: {Path(yolo).resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: {len(names)}\n'
        'names:\n' + ''.join(f'  - {n}\n' for n in names), encoding='utf-8')


def preprocess(tag, clone, variant='', names_yaml=None, reuse_labels=None, casefold_order=False):
    """parse_redraw -> build_dataset of the cloned tag, with the commands of the v1.1.0 README.

    variant / names_yaml / reuse_labels / casefold_order are only used by the diagnostic hypotheses of
    Step 5; variants never pass --write-yaml (the clone's configs/dataset.yaml stays that of the main run)."""
    rr, vid = raw_root(), f'{tag}{variant}'
    labels = Path(reuse_labels) if reuse_labels else D['data'] / f'labels_{vid}'
    yolo = D['data'] / f'yolo_{vid}'
    yaml_cfg = Path(clone) / 'configs' / 'dataset.yaml'
    parse_yaml = Path(names_yaml) if names_yaml else yaml_cfg
    pr, bd = find_script(clone, 'parse_redraw'), find_script(clone, 'build_dataset')
    pkey, bkey = f'parse_{vid}', f'build_{vid}'
    if reuse_labels:
        pinfo = {'reused_labels': str(labels), 'summary': f'labels reused from {labels.name}',
                 'effective_args': marker(f'parse_{labels.name[len("labels_"):]}').get('effective_args', {})}
    elif done(pkey):
        pinfo = marker(pkey)
        print(f'[skip] parse_redraw ({vid}) already done: {pinfo.get("summary")}')
    else:
        kill_stale(labels, ('python',))   # a parse_redraw of an interrupted earlier run
        if labels.exists():
            shutil.rmtree(labels)  # start empty: stale label files would change the split
        argv = [str(a) for a in ('--raw-root', rr, '--out-root', labels, '--dataset-yaml', parse_yaml)]
        eff = effective_args(pr, argv)
        print('parse_redraw effective arguments:', json.dumps(eff['args']))
        t0 = time.time()
        _, out = run([sys.executable, pr, *argv], cwd=clone, log=f'preprocess_{vid}.log')
        summ = [l for l in out.splitlines() if 'Total pairs' in l]
        m = re.search(r'Total pairs:\s*(\d+),\s*label files written:\s*(\d+)', out)
        pinfo = {'command': qcmd(['python', pr, *argv]), 'cwd': str(clone), 'effective_args': eff['args'],
                 'summary': summ[-1].strip() if summ else None, 'seconds': round(time.time() - t0),
                 'total_pairs': int(m.group(1)) if m else None, 'label_files_written': int(m.group(2)) if m else None,
                 'label_files': sum(1 for _ in labels.rglob('*.txt'))}
        mark(pkey, pinfo)
    if done(bkey):
        binfo = marker(bkey)
        print(f'[skip] build_dataset ({vid}) already done: {binfo.get("summary")}')
    else:
        kill_stale(yolo, ('python',))
        argv = ['--labels-root', labels, '--images-root', rr, '--yolo-root', yolo, '--clean']
        argv += ['--write-yaml', '--dataset-yaml', yaml_cfg] if not variant else ['--dataset-yaml', parse_yaml]
        argv += ['--symlink'] if USE_SYMLINKS else []
        argv = [str(a) for a in argv]
        eff = effective_args(bd, argv)
        t0 = time.time()
        cmd = [sys.executable, '-c', _CASEFOLD_BUILD, bd, *argv] if casefold_order else [sys.executable, bd, *argv]
        _, out = run(cmd, cwd=clone, log=f'preprocess_{vid}.log')
        m = re.search(r'Total pairs:\s*(\d+)\s*\|\s*train:(\d+)\s+val:(\d+)\s+test:(\d+)', out)
        mu = re.search(r'Unmatched image count[^:]*:\s*(\d+)', out)
        shown = (['python', '-c', '<build_dataset.py with case-insensitive sorted()>'] if casefold_order else ['python'])
        binfo = {'command': qcmd(shown + [bd, *argv]), 'cwd': str(clone), 'effective_args': eff['args'],
                 'constants': eff['constants'], 'summary': m.group(0) if m else None,
                 'total_pairs': int(m.group(1)) if m else None,
                 'sizes': dict(zip(SPLITS, map(int, m.groups()[1:]))) if m else None,
                 'unmatched_images': int(mu.group(1)) if mu else None, 'seconds': round(time.time() - t0),
                 'path_order': 'case-insensitive (Windows)' if casefold_order else 'python default (case-sensitive)'}
        if variant:
            write_yaml13(yolo, read_names(yaml_cfg))
        mark(bkey, binfo)
    return {'id': vid, 'tag': tag, 'variant': variant or None, 'clone': str(clone), 'labels': str(labels),
            'yolo': str(yolo), 'yaml': str(yaml_cfg if not variant else yolo / 'dataset.yaml'),
            'parse': pinfo, 'build': binfo}


def preprocessing_checks(att):
    """Rows: the arguments/constants that decide the split are those of the thesis pipeline."""
    rows = []
    a = att['parse'].get('effective_args', {})
    for k, v in PINNED['parse_redraw_defaults'].items():
        rows.append(check_row(f'parse_redraw {k}', v, a.get(k)))
    c = att['build'].get('constants', {})
    for k, v in PINNED['build_dataset_constants'].items():
        got = c.get(k)
        rows.append(check_row(f'build_dataset {k}', v, list(got) if isinstance(got, (list, tuple)) else got))
    rows.append(check_row('build_dataset unmatched images', 'reported', att['build'].get('unmatched_images'),
                          status='INFO'))
    return rows


# ---------------- scans of the raw ReDraw data (diagnostics) ----------------
def scan_raw_names(rr):
    """File names in the unpacked archive: anything except hierarchy_<n>.xml / screenshot_<n>.png, nested
    folders, and names that differ only in upper/lower case (they would behave differently on macOS/Windows)."""
    rr = Path(rr)
    apps = sorted(p for p in rr.iterdir() if p.is_dir() and p.name.endswith('-screens'))
    other, nested, case_dupes, n_xml, n_png = [], [], [], 0, 0
    for a in apps:
        seen = {}
        for f in a.iterdir():
            if f.is_dir():
                nested.append(f'{a.name}/{f.name}/')
            elif RAW_NAME_RE.match(f.name):
                n_xml += f.name.endswith('.xml')
                n_png += f.name.endswith('.png')
            else:
                other.append(f'{a.name}/{f.name}')
            k = f.name.lower()
            if k in seen:
                case_dupes.append(f'{a.name}/{seen[k]} ~ {f.name}')
            seen[k] = f.name
    lower = Counter(a.name.lower() for a in apps)
    return {'app_folders': len(apps), 'hierarchy_xml': n_xml, 'screenshot_png': n_png,
            'app_folders_with_upper_case': sum(1 for a in apps if a.name != a.name.lower()),
            'other_files': len(other), 'other_files_examples': other[:20], 'nested_dirs': nested[:20],
            'case_only_duplicate_files': case_dupes[:20], 'n_case_only_duplicate_files': len(case_dupes),
            'case_only_duplicate_app_folders': sorted(n for n, c in lower.items() if c > 1)[:20],
            'other_dirs_ignored_by_parse_redraw': [p.name for p in rr.iterdir()
                                                   if p.is_dir() and not p.name.endswith('-screens')][:20]}


def scan_raw_pairs(rr):
    """Re-counts what parse_redraw (v1.1.0) sees, independently of it: XML files, XMLs without a screenshot,
    unreadable screenshots, XML parse errors (these are skipped silently by parse_redraw) and image sizes."""
    import xml.etree.ElementTree as ET
    from PIL import Image
    res = Counter()
    sizes, parse_err, unreadable = Counter(), [], []
    for a in sorted(p for p in Path(rr).iterdir() if p.is_dir() and p.name.endswith('-screens')):
        for x in a.glob('hierarchy_*.xml'):
            res['xml_files'] += 1
            m = re.search(r'hierarchy_(\d+)\.xml$', x.name)
            img = next((x.with_name(f'screenshot_{m.group(1)}{e}') for e in IMG_EXTS
                        if m and x.with_name(f'screenshot_{m.group(1)}{e}').exists()), None)
            if img is None:
                res['xml_without_screenshot'] += 1
                continue
            try:
                with Image.open(img) as im:
                    sizes[f'{im.size[0]}x{im.size[1]}'] += 1
            except Exception:
                res['screenshot_unreadable'] += 1
                unreadable.append(str(img.relative_to(rr)))
                continue
            res['pairs'] += 1
            try:
                ET.parse(x)
            except Exception as e:
                res['xml_parse_errors'] += 1
                parse_err.append(f'{x.relative_to(rr)}: {e}')
    return {**{k: res.get(k, 0) for k in ('xml_files', 'xml_without_screenshot', 'screenshot_unreadable', 'pairs',
                                         'xml_parse_errors')},
            'image_sizes': dict(sizes.most_common(10)), 'xml_parse_error_examples': parse_err[:20],
            'unreadable_examples': unreadable[:20]}


# ---------------- dataset checks ----------------
def read_names(yaml_path):
    import yaml
    data = yaml.safe_load(Path(yaml_path).read_text(encoding='utf-8')) or {}
    names = data.get('names', [])
    return [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)


def stems_sha256(stems):  # identical to app_split split_fingerprints (sorted stems, '\n'-terminated)
    return hashlib.sha256(''.join(f'{s}\n' for s in sorted(stems)).encode('utf-8')).hexdigest()


def list_images(yolo, split):
    d = Path(yolo) / 'images' / split
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS) if d.is_dir() else []


def count_dataset(yolo, names):
    res = {}
    for s in SPLITS:
        imgs = list_images(yolo, s)
        labs = sorted((Path(yolo) / 'labels' / s).glob('*.txt'))
        inst, with_cls, empty = Counter(), Counter(), 0
        for lf in labs:
            here = Counter()
            for ln in lf.read_text(encoding='utf-8').splitlines():
                parts = ln.split()
                if len(parts) >= 5:
                    here[int(float(parts[0]))] += 1
            empty += not here
            inst.update(here)
            with_cls.update(here.keys())
        ids = sorted(set(range(len(names))) | set(inst))
        nm = lambda c: names[c] if c < len(names) else f'class_{c}'
        res[s] = {'images': len(imgs), 'label_files': len(labs), 'empty_label_files': empty,
                  'instances': sum(inst.values()),
                  'per_class_instances': {nm(c): inst.get(c, 0) for c in ids},
                  'per_class_images': {nm(c): with_cls.get(c, 0) for c in ids},
                  'stems_sha256': stems_sha256(p.stem for p in imgs),
                  'labels_match_images': sorted(p.stem for p in imgs) == sorted(p.stem for p in labs)}
    return res


def package_of(stem):
    """Package name encoded in a build_dataset image name ('<package>-screens_screenshot_<n>')."""
    m = STEM_RE.match(stem)
    if not m:
        raise PipelineStop(f'unexpected image name {stem!r} (expected <package>-screens_screenshot_<n>)')
    return m.group('app')[:-len('-screens')]


def package_stats(stems_by_split):
    """Thesis Section 4.5.5 numbers, computed from the split alone (package compared in lower case, as app_split)."""
    pk = {s: [package_of(x).lower() for x in stems_by_split[s]] for s in SPLITS}
    train_pk = set(pk['train'])
    out = {}
    for s in ('val', 'test'):
        apps = set(pk[s])
        out[s] = {'screens': len(pk[s]), 'apps': len(apps), 'apps_also_in_train': len(apps & train_pk),
                  'unseen_screens_package_only': sum(1 for p in pk[s] if p not in train_pk)}
    return out


def example_files(clone):
    """(stem, path) of the bundled examples/*.png of the repository."""
    return sorted((p.stem, p) for p in (Path(clone) / 'examples').glob('*.png'))


def example_stems(clone):
    return [s for s, _ in example_files(clone)]


def same_pixels(a, b):
    from PIL import Image
    import numpy as np
    with Image.open(a) as ia, Image.open(b) as ib:
        if ia.size != ib.size:
            return False
        return bool(np.array_equal(np.asarray(ia.convert('RGBA')), np.asarray(ib.convert('RGBA'))))


def examples_vs_raw(clone, rr):
    """Is every bundled example identical to its raw ReDraw screenshot? (Zenodo data = thesis data, and the
    YOLO name -> raw file mapping is right.) MD5 first, decoded pixels if the bytes differ."""
    res = []
    for stem, p in example_files(clone):
        m = STEM_RE.match(stem)
        raw = Path(rr) / m.group('app') / f'screenshot_{m.group("n")}.png' if m else None
        if raw is None or not raw.exists():
            res.append({'stem': stem, 'raw': None, 'md5_equal': False, 'pixels_equal': False})
            continue
        md5_eq = file_hash(p, 'md5') == file_hash(raw, 'md5')
        res.append({'stem': stem, 'raw': str(raw.relative_to(rr)), 'md5_equal': md5_eq,
                    'pixels_equal': md5_eq or same_pixels(p, raw)})
    return res


def verify_attempt(att, examples):
    c = count_dataset(att['yolo'], read_names(att['yaml']))
    stems = {s: [p.stem for p in list_images(att['yolo'], s)] for s in SPLITS}
    ps = package_stats(stems)
    test_stems = set(stems['test'])
    ex_in = [s for s in examples if s in test_stems]
    E = EXPECTED
    rows = [check_row('train screens', E['train_screens'], c['train']['images']),
            check_row('val screens', E['val_screens'], c['val']['images']),
            check_row('test screens', E['test_screens'], c['test']['images']),
            check_row('total pairs (build_dataset)', E['total_pairs'], att['build'].get('total_pairs')),
            check_row('training instances (label lines in labels/train)', E['train_instances'],
                      c['train']['instances']),
            check_row('bundled examples/*.png that are test screens', f"{E['examples_in_test']}/10",
                      f'{len(ex_in)}/{len(examples)}', ok=len(ex_in) == len(examples) == E['examples_in_test']),
            check_row('every image has a label file', True, all(c[s]['labels_match_images'] for s in SPLITS))]
    for s in SPLITS:   # Table 3.1, one row per split (the note lists every class that differs)
        exp = (EXPECTED_CLASS_COUNTS or {}).get(s)
        if not exp:
            continue
        got = c[s]['per_class_instances']
        bad = [f'{k} {got.get(k, 0):,} (thesis {v:,})' for k, v in exp.items() if got.get(k, 0) != v]
        bad += [f'{k} {v:,} (not in Table 3.1)' for k, v in got.items() if k not in exp and v]
        rows.append(check_row(f'{s} instances per class = Table 3.1', f'{len(exp)}/{len(exp)} classes',
                              f'{len(exp) - len([b for b in bad if "thesis" in b])}/{len(exp)} classes', ok=not bad,
                              note=('differ: ' + '; '.join(bad)) if bad else f'total {c[s]["instances"]:,}'))
    rows += [check_row('test screens whose package never occurs in train', E['unseen_test_package_only'],
                       ps['test']['unseen_screens_package_only']),
             check_row('test apps (distinct packages)', E['test_apps'], ps['test']['apps']),
             check_row('test apps that also occur in train', E['test_apps_in_train'], ps['test']['apps_also_in_train']),
             check_row('val screens whose package never occurs in train', 'reported',
                       ps['val']['unseen_screens_package_only'], status='INFO',
                       note=f"{ps['val']['apps']:,} val apps, {ps['val']['apps_also_in_train']:,} also in train")]
    for s in SPLITS:
        rows.append(check_row(f'{s} fingerprint (sha256 of sorted stems)', 'recorded',
                              c[s]['stems_sha256'][:16] + '...', status='INFO'))
    return all(r['status'] in ('PASS', 'INFO') for r in rows), rows, c, ex_in, ps


def class_table(counts):
    names = list(counts['train']['per_class_instances'])
    lines = [f"{'class':<14}" + ''.join(f'{s + " inst":>12}{s + " img":>11}' for s in SPLITS)]
    for n in names:
        lines.append(f'{n:<14}' + ''.join(f"{counts[s]['per_class_instances'].get(n, 0):>12,}"
                                          f"{counts[s]['per_class_images'].get(n, 0):>11,}" for s in SPLITS))
    lines.append(f"{'TOTAL':<14}" + ''.join(f"{counts[s]['instances']:>12,}{counts[s]['images']:>11,}"
                                            for s in SPLITS))
    return lines


# ---------------- REPORT.txt and the diagnostics bundle ----------------
def _verdict(checks, prefix):
    xs = [r['status'] for k, v in checks.items() if k.startswith(prefix) for r in v]
    if not xs:
        return 'not run'
    return 'FAIL' if 'FAIL' in xs else 'PASS (with WARN)' if 'WARN' in xs else 'PASS'


def key_numbers():
    """The numbers to send back, in a few lines."""
    act, ver = load_state('active') or {}, load_state('verification') or {}
    met, aps, pkg = load_state('metrics_summary') or {}, load_state('appsplit') or {}, load_state('package') or {}
    exs = load_state('examples_check') or {}
    L = []
    if act.get('counts'):
        c = act['counts']
        L.append(f"dataset used: {act.get('id')}   split reproduced: {'PASS' if act.get('split_ok') else 'FAIL'}")
        L.append(f"screens train/val/test: {c['train']['images']:,} / {c['val']['images']:,} / {c['test']['images']:,}"
                 f"   (thesis 14,955 / 1,869 / 1,870)")
        L.append(f"instances train/val/test: {c['train']['instances']:,} / {c['val']['instances']:,} / "
                 f"{c['test']['instances']:,}   (thesis 172,732 / 21,613 / 23,046)")
        a0 = next((a for a in ver.get('attempts', []) if a['id'] == act.get('id')), {})
        if a0.get('package_stats'):
            t = a0['package_stats']['test']
            L.append(f"test screens unseen by package: {t['unseen_screens_package_only']:,} (thesis 214)   "
                     f"test apps: {t['apps']:,}, also in train: {t['apps_also_in_train']:,} (thesis 1,595 / 1,392)")
        L.append(f"examples in test: {len(a0.get('examples_in_test', []))}/{len(ver.get('examples', []))}")
    if exs:
        L.append(f"examples identical to the raw ReDraw screenshot: pixels {exs.get('pixels_equal')}/{exs.get('n')}, "
                 f"bytes (MD5) {exs.get('md5_equal')}/{exs.get('n')}")
    for k, v in sorted((met.get('values') or {}).items()):
        L.append(f"{k}: mAP@50 {v['mAP50']:.5f}  mAP@50-95 {v['mAP50_95']:.5f}")
    h = aps.get('headline') or {}
    if h:
        L.append(f"unseen test (app_split): full grouping {h.get('unseen_test_full')}, package+MD5 "
                 f"{h.get('unseen_test_package_md5')}, package only {h.get('unseen_test_package_only')}   "
                 f"unseen val: full {h.get('unseen_val_full')}, package only {h.get('unseen_val_package_only')}")
        px, md = h.get('pixel_identical_to_train') or {}, h.get('md5_identical_to_train') or {}
        L.append(f"val/test pixel-identical to a train screen: {px.get('val')} / {px.get('test')} (thesis 9 / 8); "
                 f"byte-identical (MD5): {md.get('val')} / {md.get('test')}")
    if pkg:
        L.append(f"zip: {pkg.get('name')}  {pkg.get('size_human')}  sha256 {pkg.get('sha256')}")
        L.append(f"zip on Drive: {pkg.get('drive_path') or 'NOT COPIED'}  (read back OK: {pkg.get('drive_sha256_ok')})")
    return L


def write_report(final=False):
    """Collects every PASS/FAIL table into REPORT.txt (local copy + Drive copy). Returns (text, drive_path)."""
    L = ['ui2compose - ReDraw split rebuild - REPORT', f'written: {now()}   DRY_RUN={DRY_RUN}   FORCE={FORCE}', '']
    checks = load_state('checks', {})
    act = load_state('active') or {}
    met = load_state('metrics_summary') or {}
    pkg = load_state('package') or {}
    L += ['OVERALL',
          f"  download + MD5 (Step 3):            {_verdict(checks, '3 ')}",
          f"  code = pinned v1.1.0 (Step 4):      {_verdict(checks, '4 code')}",
          f"  split reproduced (Step 5):          {('PASS' if act.get('split_ok') else 'FAIL') if act else 'not run'}"
          + (f"  -> dataset used: {act.get('id')}" if act else ''),
          f"  examples = raw ReDraw (Step 5):     {_verdict(checks, '5 examples')}",
          f"  published metrics (Step 6):         {('PASS' if met.get('ok') else 'FAIL') if met else 'not run'}",
          f"  thesis cross-checks (Step 7):       {_verdict(checks, '7 ')}",
          f"  zip on Drive (Step 8):              {pkg.get('drive_path') or 'not built'}", '']
    kn = key_numbers()
    if kn:
        L += ['KEY NUMBERS (send back the whole file; if that is not possible, copy at least this block)']
        L += ['  ' + x for x in kn] + ['']
    env = load_state('env') or {}
    if env:
        L += ['ENVIRONMENT', '  ' + json.dumps(env, default=str), '']
    for tag in [REPO_TAG] + list(FALLBACK_TAGS):
        ci = marker(f'clone_{tag}')
        if ci:
            L.append(f"CODE {tag}: commit {ci.get('commit')}  " + ('(pinned hashes OK)' if not ci.get('problems')
                                                                    else 'PROBLEMS: ' + '; '.join(ci['problems'])))
    for key in ('parse', 'build'):
        for f in sorted(D['markers'].glob(f'{key}_*.done')):
            i = jload(f, {})
            L.append(f"  {f.stem}: {i.get('command')}")
            L.append(f"      -> {i.get('summary')}")
    L.append('')
    for sec, rows in sorted(checks.items(), key=lambda kv: int(kv[0].split()[0])):   # by step number (stable)
        L += table_lines(sec, rows) + ['']
    if act.get('counts'):
        L += [f"PER-CLASS COUNTS ({act.get('id')})"] + class_table(act['counts']) + ['']
    raw = {k: marker(k) for k in ('rawscan_names', 'rawscan_pairs') if done(k)}
    if raw:
        L += ['RAW DATA SCANS', '  ' + json.dumps(raw, default=str), '']
    L.append('final report' if final else 'interim report (later steps not finished yet)')
    text = '\n'.join(L) + '\n'
    (D['out'] / 'REPORT.txt').write_text(text, encoding='utf-8')
    dest = None
    if drive_alive():
        try:
            dest = Path(OUT_DRIVE_DIR) / 'REPORT.txt'
            dest.write_text(text, encoding='utf-8')
        except OSError as e:
            print(f'WARNING: REPORT.txt could not be written to Google Drive ({e}); local copy: {D["out"] / "REPORT.txt"}')
            dest = None
    return text, dest


def write_diagnostics():
    """Small zip (no images, ~5-20 MB) for diagnosis, refreshed after Steps 5-7 and before a FAIL stop:
    REPORT.txt, all state/markers/logs, the split stem lists of every attempt, the labels of all splits of the
    dataset in use, the metric JSONs and the app_split outputs. Copied to Drive next to REPORT.txt."""
    text, _ = write_report()
    act = load_state('active') or load_state('primary') or {}
    path = D['out'] / DIAG_ZIP
    tmp = path.with_name(path.name + '.partial')

    def add_text(zf, arc, txt):
        zi = zipfile.ZipInfo(f'ui2compose_diagnostics/{arc}', date_time=time.localtime()[:6])
        zi.external_attr = 0o644 << 16
        zf.writestr(zi, txt, compress_type=zipfile.ZIP_DEFLATED)

    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        add_text(zf, 'REPORT.txt', text)
        for sub in ('state', 'markers', 'logs', 'metrics'):
            for p in sorted(D[sub].iterdir()):
                if p.is_file() and not p.name.endswith('.tmp'):
                    zf.write(p, f'ui2compose_diagnostics/{sub}/{p.name}')
        for yolo in sorted(D['data'].glob('yolo_*')):
            for s in SPLITS:
                add_text(zf, f'splits/{yolo.name[5:]}/split_{s}_stems.txt',
                         ''.join(p.stem + '\n' for p in list_images(yolo, s)))
        if act.get('yolo'):
            for s in SPLITS:
                for p in sorted((Path(act['yolo']) / 'labels' / s).glob('*.txt')):
                    zf.write(p, f'ui2compose_diagnostics/labels_{act["id"]}/{s}/{p.name}')
        for p in sorted(D['splits'].rglob('*')):
            if p.is_file() and p.suffix in ('.csv', '.json', '.txt'):
                zf.write(p, f'ui2compose_diagnostics/app_split/{p.relative_to(D["splits"]).as_posix()}')
    os.replace(tmp, path)
    dest, _ = copy_to_drive(path)
    print(f'diagnostics bundle: {path} ({human(path.stat().st_size)})'
          + (f' -> {dest}' if dest else ' (not copied to Google Drive)'))
    return path, dest


print('helpers ready; work directory', WORK)

## Step 2 – Google Drive, environment, packages

**Google Drive first:** a few seconds after *Run all*, a pop-up asks for access to your Google Drive. Choose your Google account and allow access (you have about 2 minutes). The zip, `REPORT.txt` and a small diagnostics zip are written to `MyDrive/ui2compose_revision/`. The notebook stops right away if your Drive has less than 3.5 GB free, so that you can free up space before the long part starts.

Then it checks the free disk of the Colab machine (it stops if less than 40 GB is free), shows the Python version and whether a GPU is present (a GPU is **not** needed), and runs pip with `ultralytics==8.4.24`, the version the released weights were trained with, plus the other requirements of v1.1.0 (`opencv-python>=4.8.0`, `Pillow>=10.0.0`, `numpy>=1.24.0`, `PyYAML>=6.0`, `matplotlib>=3.7.0`). `streamlit` is left out because only the demo UI uses it. pip only adds what is missing, so packages Colab already has, including its `torch`, are not upgraded; the table confirms that `torch` was not changed.

From the end of this step on, `REPORT.txt` on Drive is updated after every step.

In [ ]:
#@title Step 2 - Connect Google Drive, check disk / Python / GPU, install Ultralytics
import importlib.metadata as _md

def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

# 1) Google Drive first, so that the pop-up appears within seconds of "Run all"
mount_drive()
drive_free = drive_free_gb()
drive_need = DRIVE_MIN_FREE_GB + (12 if (TARBALL_DIR and str(TARBALL_DIR).startswith('/content/drive')
                                        and not done('download')) else 0)
print(f'Google Drive free space: ' + (f'{drive_free:.1f} GB' if drive_free is not None else 'not reported')
      + f' (need about {drive_need:.1f} GB)')
if drive_free is not None and drive_free < drive_need and not DRY_RUN:
    raise PipelineStop(f'your Google Drive has only {drive_free:.1f} GB free, but the zip and the report need about '
                       f'{drive_need:.1f} GB. Free up space on drive.google.com (and empty its bin), then choose '
                       'Run all again.')

# 2) local disk, Python, GPU
free = free_gb(WORK)
print(f'Python {sys.version.split()[0]} on {platform.platform()}')
print(f'free disk at {WORK}: {free:.1f} GB (need >= {MIN_FREE_GB} GB)')
if free < MIN_FREE_GB and not (done('extract') and free > 15):
    raise PipelineStop(f'only {free:.1f} GB free disk at {WORK}; this notebook needs {MIN_FREE_GB} GB. '
                       'Use a fresh Colab runtime (Runtime -> Disconnect and delete runtime).')
gpu_name = None
try:
    gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                              capture_output=True, text=True, timeout=30).stdout.strip() or None
except Exception:
    pass
print('GPU:', gpu_name or f'none - CPU runtime ({os.cpu_count()} CPUs). This is fine: only the metric check in '
      'Step 6 is slower (~40-55 min instead of ~10 min).')

# 3) ultralytics pinned to the training version + the other requirements of v1.1.0 (requirements.txt without
# streamlit, which only the demo UI needs). pip installs only what is missing; nothing that is already
# satisfied (e.g. Colab's torch, numpy) is upgraded.
REQUIREMENTS = [f'ultralytics=={ULTRALYTICS_VERSION}', 'opencv-python>=4.8.0', 'Pillow>=10.0.0', 'numpy>=1.24.0',
                'PyYAML>=6.0', 'matplotlib>=3.7.0']
torch_before = _ver('torch')
if DRY_RUN:
    print(f'DRY_RUN: not installing anything (ultralytics {_ver("ultralytics")} found)')
elif not done('pip') or _ver('ultralytics') != ULTRALYTICS_VERSION:
    run([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS], log='pip.log')
    mark('pip', {'requirements': REQUIREMENTS})
torch_after = _ver('torch')
env = {'python': sys.version.split()[0], 'platform': platform.platform(), 'cpus': os.cpu_count(),
       'gpu': gpu_name, 'free_disk_gb': round(free, 1), 'drive_free_gb': round(drive_free, 1) if drive_free else None,
       'torch': torch_after, 'torch_before_pip': torch_before,
       'ultralytics': _ver('ultralytics'), 'numpy': _ver('numpy'), 'Pillow': _ver('Pillow') or _ver('pillow'),
       'PyYAML': _ver('PyYAML'), 'opencv': _ver('opencv-python') or _ver('opencv-python-headless'),
       'git': run(['git', '--version'], echo=False, quiet=True)[1].strip(), 'started': now()}
save_state('env', env)
rows = [check_row('Google Drive connected', True, drive_alive()),
        check_row('Google Drive free (GB)', f'>= {drive_need:.1f}', round(drive_free, 1) if drive_free is not None
                  else 'not reported', status='PASS' if drive_free is not None and drive_free >= drive_need else 'WARN'),
        check_row('free disk (GB)', f'>= {MIN_FREE_GB}', round(free, 1), ok=free >= MIN_FREE_GB or done('extract')),
        check_row('ultralytics version', ULTRALYTICS_VERSION, env['ultralytics']),
        check_row('torch unchanged by pip', torch_before, torch_after),
        check_row('GPU', 'not needed (CPU is fine)', gpu_name or 'none', status='INFO')]
set_checks('2 environment', rows)
if env['ultralytics'] != ULTRALYTICS_VERSION:
    raise PipelineStop(f'ultralytics {env["ultralytics"]} is installed, {ULTRALYTICS_VERSION} is needed. '
                       'Choose Runtime -> Restart session, then Run all again.')
write_report()
print('\nStep 2 finished. From here on the notebook runs by itself; keep this tab open.')

## Step 3 – Download ReDraw from Zenodo, verify, unpack

* The Zenodo API (`https://zenodo.org/api/records/2530277`) gives the download link, the size (11,579,793,021 bytes) and the MD5 (`8270e4aa66c3b86f495790dd64cf515b`) of `ReDraw-Final-Google-Play-Dataset.tar.gz`.
* The download resumes after an interruption (`curl -C -`, up to 40 attempts). Speed and ETA are printed every minute. **This is the slow part: about 1–3 hours** at the 1.4–2.3 MB/s measured from Zenodo (Colab may be faster). Check the tab every 30–60 minutes and answer Colab's "Are you still there?" dialog if it appears.
* If you interrupt the cell, the download process is stopped too; *Run all* continues where it stopped. A download process left over from an earlier session is stopped before a new one starts.
* After the download, the MD5 is checked. If it does not match, the file is deleted and the notebook stops; *Run all* then downloads it again.
* Layout of the archive, from a probe of its first 400 MB: one top folder `ReDraw-Final-Google-Play-Dataset/` that holds 6,526 app folders `<package>-screens/`, each with `hierarchy_<n>.xml` (uiautomator dump) and `screenshot_<n>.png` (1200×1920). This is the `--raw-root` that `parse_redraw.py` expects; it reads every `*-screens` folder directly below the raw root.
* After unpacking (5–10 min), every file name is checked (only `hierarchy_<n>.xml` / `screenshot_<n>.png` are expected, and no two names that differ only in upper/lower case), and the archive is deleted to free 11.6 GB.

In [ ]:
#@title Step 3a - Download (resumable) and MD5 check
import urllib.request

def zenodo_file_info():
    if DRY_RUN and not (DRY_RUN_RECORD_JSON or DRY_RUN_ARCHIVE):
        raise PipelineStop('DRY_RUN needs a synthetic archive: set the environment variable UI2C_DRYRUN_ARCHIVE '
                           '(or UI2C_DRYRUN_RECORD_JSON); untick DRY_RUN for the real run')
    if DRY_RUN and DRY_RUN_RECORD_JSON:
        rec, src = json.loads(Path(DRY_RUN_RECORD_JSON).read_text()), DRY_RUN_RECORD_JSON
    elif DRY_RUN and DRY_RUN_ARCHIVE:
        p = Path(DRY_RUN_ARCHIVE)
        rec, src = {'files': [{'key': ZENODO_FILE, 'size': p.stat().st_size, 'links': {'self': p.resolve().as_uri()},
                               'checksum': 'md5:' + file_hash(p, 'md5')}]}, str(p)
    else:
        src = f'https://zenodo.org/api/records/{ZENODO_RECORD}'
        for attempt in range(1, 6):
            try:
                with urllib.request.urlopen(src, timeout=60) as r:
                    rec = json.loads(r.read().decode('utf-8'))
                break
            except Exception as e:
                print(f'Zenodo API attempt {attempt} failed: {e}')
                if attempt == 5:
                    raise PipelineStop(f'cannot reach the Zenodo API ({src}); try again later')
                time.sleep(20 * attempt)
    f = [x for x in rec.get('files', []) if x.get('key') == ZENODO_FILE]
    if not f:
        raise PipelineStop(f'{ZENODO_FILE} not listed in {src}')
    f = f[0]
    algo, digest = f['checksum'].split(':', 1)
    return {'record': ZENODO_RECORD, 'api': src, 'key': f['key'], 'size': int(f['size']),
            'url': f['links']['self'], 'checksum_algo': algo, 'checksum': digest}


def download_with_resume(url, dest, size, max_attempts=40):
    dest.parent.mkdir(parents=True, exist_ok=True)
    kill_stale(dest, ('curl',))   # a curl of an earlier session (e.g. after a kernel restart) writing the same file
    every = 1 if DRY_RUN else 60
    for attempt in range(1, max_attempts + 1):
        have = dest.stat().st_size if dest.exists() else 0
        if have == size:
            return
        if have > size:
            print(f'{dest} is larger than expected ({have} > {size}); starting again')
            dest.unlink()
            have = 0
        print(f'download attempt {attempt}: {human(have)} of {human(size)} already on disk', flush=True)
        p = subprocess.Popen(['curl', '-L', '-sS', '--fail', '-C', '-', '--connect-timeout', '60',
                              '--speed-limit', '10240', '--speed-time', '180', '-o', str(dest), url],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        t_last, b_last = time.time(), have
        try:
            while True:
                try:
                    p.wait(timeout=every)
                    break
                except subprocess.TimeoutExpired:
                    b = dest.stat().st_size if dest.exists() else 0
                    rate = (b - b_last) / max(time.time() - t_last, 1e-9)
                    t_last, b_last = time.time(), b
                    eta = (size - b) / rate / 60 if rate > 0 else float('inf')
                    print(f'  {human(b)} / {human(size)} ({100 * b / size:.1f} %)  {rate / 1e6:.1f} MB/s  '
                          f'ETA {eta:.0f} min', flush=True)
        except BaseException:   # interrupted: stop curl too, so a later Run all does not start a second one
            p.kill()
            p.wait()
            print('download interrupted; Run all continues where it stopped')
            raise
        msg = p.stdout.read().strip()
        if p.returncode != 0:
            print(f'  curl stopped (exit {p.returncode}): {msg[-300:]}; retrying in {min(120, 15 * attempt)} s')
            time.sleep(0 if DRY_RUN else min(120, 15 * attempt))
    if not dest.exists() or dest.stat().st_size != size:
        raise PipelineStop('the download did not complete after many attempts; Run all again later '
                           '(it resumes where it stopped)')


if done('extract'):
    zinfo = load_state('zenodo')
    print(f'[skip] archive already downloaded, verified and unpacked ({marker("extract").get("raw_root")})')
else:
    zinfo = zenodo_file_info()
    tar_path = Path(TARBALL_DIR or D['download']) / ZENODO_FILE
    zinfo['local_path'] = str(tar_path)
    save_state('zenodo', zinfo)
    print(f"{zinfo['key']}: {zinfo['size']:,} bytes, {zinfo['checksum_algo']} {zinfo['checksum']}\n  from {zinfo['url']}")
    if not done('download'):
        have = tar_path.stat().st_size if tar_path.exists() else 0
        need = (zinfo['size'] - have) / 1e9 * (0 if TARBALL_DIR else 1) + zinfo['size'] / 1e9 * 1.05 + 2
        if free_gb(WORK) < need:
            raise PipelineStop(f'not enough disk for download + unpacking: {free_gb(WORK):.1f} GB free, '
                               f'~{need:.0f} GB needed')
        t0 = time.time()
        download_with_resume(zinfo['url'], tar_path, zinfo['size'])
        mark('download', {'path': str(tar_path), 'size': zinfo['size'], 'minutes': round((time.time() - t0) / 60, 1)})
    if not done('md5'):
        print(f'checking {zinfo["checksum_algo"]} of {tar_path} ...', flush=True)
        got = file_hash(tar_path, zinfo['checksum_algo'], progress_every=0 if DRY_RUN else 30)
        ok = got == zinfo['checksum']
        set_checks('3 download', [check_row(f'{zinfo["checksum_algo"]} of {ZENODO_FILE}', zinfo['checksum'], got),
                                  check_row('size (bytes)', zinfo['size'], tar_path.stat().st_size)])
        if not ok:
            tar_path.unlink()
            for k in ('download',):
                (D['markers'] / f'{k}.done').unlink(missing_ok=True)
            write_report()
            raise PipelineStop('MD5 mismatch: the downloaded file was deleted. Choose Run all to download it again.')
        mark('md5', {'md5': got})
    else:
        print('[skip] MD5 already verified:', marker('md5').get('md5'))
    write_report()

In [ ]:
#@title Step 3b - Unpack and check the layout and the file names
if done('extract'):
    xinfo = marker('extract')
    print(f'[skip] already unpacked: {xinfo.get("raw_root")} ({xinfo.get("app_folders"):,} app folders)')
else:
    zinfo = load_state('zenodo')
    tar_path = Path(zinfo['local_path'])
    if not done('md5') or not tar_path.exists():
        raise PipelineStop('the verified archive is missing - run Step 3a first')
    dest = D['raw']
    kill_stale(tar_path, ('tar', 'gzip', 'pigz'))   # an unpack of an earlier session that is still running
    if dest.exists():
        shutil.rmtree(dest)  # a previous unpack was interrupted: start clean
    dest.mkdir(parents=True)
    t0 = time.time()
    print(f'unpacking {tar_path} -> {dest} (5-10 min) ...', flush=True)
    run(['tar', '--no-same-owner', '-xzf', tar_path, '-C', dest], log='extract.log')
    tops = [p for p in dest.iterdir() if p.is_dir()]
    cand = [p for p in tops if any(c.is_dir() and c.name.endswith('-screens') for c in p.iterdir())]
    if (dest / 'ReDraw-Final-Google-Play-Dataset').is_dir():
        rr = dest / 'ReDraw-Final-Google-Play-Dataset'
    elif len(cand) == 1:
        rr = cand[0]
    elif any(c.name.endswith('-screens') for c in tops):
        rr = dest
    else:
        raise PipelineStop(f'unexpected archive layout in {dest}: {[p.name for p in tops][:10]}')
    print('checking every file name ...', flush=True)
    names = scan_raw_names(rr)
    mark('rawscan_names', names)
    xinfo = {'raw_root': str(rr), 'app_folders': names['app_folders'], 'hierarchy_xml': names['hierarchy_xml'],
             'screenshots': names['screenshot_png'], 'minutes': round((time.time() - t0) / 60, 1)}
    rows = [check_row('raw root (parse_redraw --raw-root)', 'ReDraw-Final-Google-Play-Dataset', rr.name,
                      status='PASS' if rr.name == 'ReDraw-Final-Google-Play-Dataset' or DRY_RUN else 'WARN'),
            check_row('app folders <package>-screens', EXPECTED['app_folders_in_archive'], names['app_folders'],
                      status='PASS' if names['app_folders'] == EXPECTED['app_folders_in_archive'] else 'WARN',
                      note='from the archive probe; informational'),
            check_row('hierarchy_<n>.xml files', 'reported', names['hierarchy_xml'], status='INFO'),
            check_row('screenshot_<n>.png files', 'reported', names['screenshot_png'], status='INFO'),
            check_row('other file names inside app folders', 0, names['other_files'],
                      status='PASS' if not names['other_files'] else 'WARN',
                      note=', '.join(names['other_files_examples'][:5])),
            check_row('sub-folders inside app folders', 0, len(names['nested_dirs']),
                      status='PASS' if not names['nested_dirs'] else 'WARN', note=', '.join(names['nested_dirs'][:5])),
            check_row('file names differing only in upper/lower case', 0, names['n_case_only_duplicate_files'],
                      status='PASS' if not names['n_case_only_duplicate_files'] else 'WARN',
                      note=', '.join(names['case_only_duplicate_files'][:5])),
            check_row('app folders differing only in upper/lower case', 0,
                      len(names['case_only_duplicate_app_folders']),
                      status='PASS' if not names['case_only_duplicate_app_folders'] else 'WARN',
                      note=', '.join(names['case_only_duplicate_app_folders'][:5])),
            check_row('app folders with upper-case letters', 'reported', names['app_folders_with_upper_case'],
                      status='INFO', note='probe: 671'),
            check_row('other folders next to the app folders', 'ignored by parse_redraw',
                      ', '.join(names['other_dirs_ignored_by_parse_redraw']) or 'none', status='INFO')]
    set_checks('3 extraction', rows)
    mark('extract', xinfo)
    if DELETE_TARBALL_AFTER_EXTRACT and not TARBALL_DIR:
        tar_path.unlink()
        print(f'deleted {tar_path} to free disk space')
    write_report()
print(f'free disk now: {free_gb(WORK):.1f} GB')

## Step 4 – Rebuild the split with the v1.1.0 code (10–15 min)

The repository is cloned at tag `v1.1.0`, and the commit and the SHA-256 of `parse_redraw.py`, `build_dataset.py`, `configs/dataset.yaml` and `best.pt` are compared with the values pinned in Step 1b. The two steps of the README section "Preparing the dataset" are then run. The scripts are called by path, `python src/ui2compose/preprocess/<script>.py` in the clone, which is what the `ui2compose-parse-redraw` / `ui2compose-build-dataset` entry points execute.

```
parse_redraw.py  --raw-root <ReDraw root> --out-root <labels> --dataset-yaml configs/dataset.yaml
build_dataset.py --labels-root <labels> --images-root <ReDraw root> --yolo-root <yolo> \
                 --clean --write-yaml --dataset-yaml configs/dataset.yaml --symlink
```

**What decides the split.**
* The 13-class list in `configs/dataset.yaml` (whitelist and ids).
* The `parse_redraw` defaults: `--min-area 0.00005`, `--dedup-iou 0.85`, `--drop-container true`, `--limit 0`. The internal function `extract_labels()` has other defaults (0.00010 / 0.95), but `main()` always passes the CLI values.
* `build_dataset`'s constants `SEED=42` and `SPLIT=(0.8, 0.1, 0.1)`.
* The order of the (label, image) pairs: sorted app folders, then sorted label files. Python sorts paths case-sensitively on Linux and macOS.
* An empty labels folder: label files left over from an earlier run would change the pair list, so the folder is wiped before parsing.

`--symlink` only links the images instead of copying them. The split, the labels and the file names are identical, and it saves 11 GB. The notebook prints the effective arguments and checks them in a table. Oversampling (`oversample_dataset.py`) is **not** run, because it only adds copies to the training split.

After the two scripts, the notebook recounts independently what `parse_redraw` saw: XML/screenshot pairs (must equal its "Total pairs"), XMLs without a screenshot, unreadable screenshots and XML parse errors. `parse_redraw` skips the last three silently, so these numbers help if the split does not match.

In [ ]:
#@title Step 4 - Clone v1.1.0 and run parse_redraw -> build_dataset
clone, cinfo = clone_repo(REPO_TAG)
code_rows = [check_row(f'{REPO_TAG} commit', PINNED['commits'].get(REPO_TAG), cinfo['commit'])]
code_rows += [check_row(f'{REPO_TAG} sha256 {rel}', PINNED['files'][REPO_TAG][rel][:16] + '...',
                        (h or 'missing')[:16] + '...', ok=h == PINNED['files'][REPO_TAG][rel])
              for rel, h in cinfo['file_sha256'].items()]
set_checks(f'4 code {REPO_TAG}', code_rows)
primary = preprocess(REPO_TAG, clone)
set_checks(f'4 preprocessing {primary["id"]}', preprocessing_checks(primary))
save_state('primary', primary)

# Independent recount of what parse_redraw saw (it skips unreadable files and XML parse errors silently)
if done('rawscan_pairs'):
    scan = marker('rawscan_pairs')
else:
    print('\nrecounting the XML/screenshot pairs (about 1 min) ...', flush=True)
    scan = scan_raw_pairs(raw_root())
    mark('rawscan_pairs', scan)
pp = primary['parse']
no_label = (scan['pairs'] - scan['xml_parse_errors'] - pp['label_files_written']
            if pp.get('label_files_written') is not None else None)
set_checks('4 raw data recount', [
    check_row('XML/screenshot pairs (recount = parse_redraw "Total pairs")', pp.get('total_pairs'), scan['pairs'],
              status='PASS' if pp.get('total_pairs') == scan['pairs'] else 'WARN'),
    check_row('hierarchy XML files', 'reported', scan['xml_files'], status='INFO'),
    check_row('XML without screenshot (skipped)', 'reported', scan['xml_without_screenshot'], status='INFO'),
    check_row('unreadable screenshots (skipped)', 'reported', scan['screenshot_unreadable'], status='INFO',
              note=', '.join(scan['unreadable_examples'][:3])),
    check_row('XML parse errors (skipped)', 'reported', scan['xml_parse_errors'], status='INFO',
              note=', '.join(scan['xml_parse_error_examples'][:3])),
    check_row('label files written by parse_redraw', 'reported', pp.get('label_files_written'), status='INFO'),
    check_row('pairs without any kept label (no label file)', 'reported', no_label, status='INFO'),
    check_row('screenshot sizes', 'reported', json.dumps(scan['image_sizes']), status='INFO')])
write_report()
print('\nlabels:', primary['labels'], '\nYOLO dataset:', primary['yolo'], '\ndataset.yaml:', primary['yaml'])

## Step 5 – Is it the thesis split? (PASS/FAIL)

Every check below must PASS. All of them follow from the split alone:
* 14,955 / 1,869 / 1,870 screens and 18,694 pairs (Section 3.2.3, Table A.2), and a label file for every image;
* 172,732 training instances (label lines in `labels/train`);
* the instances **per class and per split** of **Table 3.1** (Section 3.2.1; totals 172,732 / 21,613 / 23,046). This catches splits that differ only by swaps between val and test or by label changes that keep the training total;
* all 10 bundled `examples/*.png` in test;
* the application-level numbers of Section 4.5.5: 214 test screens whose package never occurs in training, 1,595 test apps, 1,392 of them also in training. The package is the part of the file name before `-screens`, compared in lower case as `app_split` does.

The per-class instance and image counts of every split are printed for the record. The repository's `results/` folder stores only metrics, no per-class instance counts.

A separate table compares each bundled `examples/*.png` with its raw ReDraw screenshot (MD5, and decoded pixels if the bytes differ). If they match, the Zenodo archive is the data the thesis used and the mapping from YOLO file names to raw files is right. A mismatch is reported as WARN; it does not block the zip.

**If a check fails**, the notebook automatically:
1. repeats Step 4 with each fallback tag (`v1.0.2`, `v1.0.1`): fresh clone, same raw data, new output folders. Their `parse_redraw.py` / `build_dataset.py` contain the same split logic as v1.1.0, so they are expected to give the same result.
2. tests diagnostic variants of the **same v1.1.0 code**, each checked in the same way:
   * **A**: the class list in v1.1.0 `src/ui2compose/preprocess/oversample_dataset.py` (line 16) is commented "13 class — remapped, CardView & Toolbar removed", and the results files say "13 classes (remapped)". If the split was made from 15-class labels and those two classes were removed afterwards, screens whose only labels were CardView/Toolbar stayed in the split as background images. A builds that: 15-class parse, unchanged split code, CardView/Toolbar lines dropped afterwards.
   * **B**: Python sorts paths case-sensitively on Linux/macOS but case-insensitively on Windows, and 671 ReDraw app folders contain upper-case letters. If the thesis split was built on Windows, the shuffle input order differs. B runs the unchanged `build_dataset.py` with the Windows path order.
   * **A+B**: both together.

The first variant that passes becomes the dataset used by Steps 6–8. If none passes, Steps 6–7 still run on the v1.1.0 result for diagnosis, but Step 8 does not build the zip unless `FORCE` is ticked. Each extra variant takes 2–10 min.

At the end of this step a small `ui2compose_diagnostics.zip` (labels of all splits, split lists, all tables and logs; no images) is written to Drive, whether the checks pass or fail, and it is refreshed after Steps 6 and 7.

In [ ]:
#@title Step 5 - Verify the split (with automatic fallbacks)
primary = load_state('primary')
if not primary:
    raise PipelineStop('Step 4 has not finished - run it first')
examples = example_stems(repo_dir(REPO_TAG))
attempts, active, active_counts = [], None, None
save_state('checks', {k: v for k, v in load_state('checks', {}).items() if not k.startswith('5 ')})  # fresh Step 5 tables


def try_attempt(att, label):
    ok, rows, counts, ex_in, ps = verify_attempt(att, examples)
    set_checks(f'5 split check {att["id"]}', rows)
    attempts.append({'id': att['id'], 'label': label, 'ok': ok, 'counts': counts, 'examples_in_test': ex_in,
                     'package_stats': ps, 'parse': att['parse'], 'build': att['build'],
                     'clone_commit': marker(f'clone_{att["tag"]}').get('commit')})
    return ok, counts


ok, counts = try_attempt(primary, f'{REPO_TAG} (README commands)')
if ok:
    active, active_counts = primary, counts
else:
    banner(f'{REPO_TAG} did NOT reproduce the thesis split - trying the fallback tags {FALLBACK_TAGS}', ok=False)
    for tag in FALLBACK_TAGS:
        fclone, finfo = clone_repo(tag)
        att = preprocess(tag, fclone)
        set_checks(f'4 preprocessing {att["id"]}', preprocessing_checks(att))
        ok, counts = try_attempt(att, f'fallback tag {tag}')
        if ok:
            active, active_counts = att, counts
            break


def hypothesis(tag, clone, fifteen, casefold):
    """Diagnostic variants of the same v1.1.0 code (only run if no tag reproduces the split).

    fifteen:  parse with 15 classes (+CardView, Toolbar), split, then drop those two classes from the labels
    casefold: build_dataset with the case-insensitive (Windows) path order"""
    suffix = ('_hyp15' if fifteen else '') + ('_casefold' if casefold else '')
    names13 = read_names(Path(clone) / 'configs' / 'dataset.yaml')
    y15, reuse = None, None
    if fifteen:
        names15 = sorted(set(names13) | {'CardView', 'Toolbar'})
        y15 = D['data'] / 'hypothesis_15class_names.yaml'
        y15.write_text('names:\n' + ''.join(f'  - {n}\n' for n in names15), encoding='utf-8')
        if casefold and done(f'parse_{tag}_hyp15'):
            reuse = D['data'] / f'labels_{tag}_hyp15'
    else:
        reuse = D['data'] / f'labels_{tag}'           # the 13-class labels of the main run
    att = preprocess(tag, clone, variant=suffix, names_yaml=y15, reuse_labels=reuse, casefold_order=casefold)
    if fifteen:
        yolo, labels = Path(att['yolo']), Path(att['labels'])
        rkey = f'remap_{att["id"]}'
        if not done(rkey):
            to13 = {i: names13.index(n) for i, n in enumerate(names15) if n in names13}
            dropped = Counter()
            for lf in yolo.glob('labels/*/*.txt'):  # rewritten from the untouched parse output -> safe to repeat
                m = STEM_RE.match(lf.stem)
                src = labels / m.group('app') / f'screenshot_{m.group("n")}.txt'
                out = []
                for ln in src.read_text(encoding='utf-8').splitlines():
                    parts = ln.split()
                    if len(parts) < 5:
                        continue
                    c = int(parts[0])
                    if c in to13:
                        out.append(' '.join([str(to13[c])] + parts[1:]))
                    else:
                        dropped[names15[c]] += 1
                lf.write_text(''.join(l + '\n' for l in out), encoding='utf-8')
            mark(rkey, {'dropped_instances': dict(dropped), 'names15': names15})
        att['remap'] = marker(rkey)
        print('CardView/Toolbar instances removed after the split:', att['remap'].get('dropped_instances'))
    return att


HYPOTHESES = [  # (fifteen, casefold, label)
    (True, False, 'hypothesis A: 15-class labels, CardView/Toolbar removed after the split'),
    (False, True, 'hypothesis B: case-insensitive (Windows) path order in build_dataset'),
    (True, True, 'hypothesis A+B'),
]
if active is None and TRY_HYPOTHESES:
    for fifteen, casefold, label in HYPOTHESES:
        banner(f'no tag reproduced the split - testing {label} (diagnostic)', ok=False)
        att = hypothesis(REPO_TAG, repo_dir(REPO_TAG), fifteen, casefold)
        ok, counts = try_attempt(att, f'{label} ({REPO_TAG} code)')
        if ok:
            active, active_counts = att, counts
            break

split_ok = active is not None
if not split_ok:
    active, active_counts = primary, attempts[0]['counts']
save_state('active', {**active, 'split_ok': split_ok, 'counts': active_counts})
save_state('verification', {'written': now(), 'split_ok': split_ok, 'active': active['id'], 'examples': examples,
                            'expected': EXPECTED, 'expected_class_counts': EXPECTED_CLASS_COUNTS,
                            'attempts': attempts})

# The bundled examples against the raw ReDraw screenshots (independent of the split)
exr = examples_vs_raw(repo_dir(REPO_TAG), raw_root())
n_px, n_md5 = sum(r['pixels_equal'] for r in exr), sum(r['md5_equal'] for r in exr)
save_state('examples_check', {'n': len(exr), 'pixels_equal': n_px, 'md5_equal': n_md5, 'details': exr})
missing = [r['stem'] for r in exr if not r['raw']]
differ = [r['stem'] for r in exr if r['raw'] and not r['pixels_equal']]
set_checks('5 examples vs raw ReDraw', [
    check_row('examples/*.png identical to their raw screenshot (decoded pixels)', '10/10', f'{n_px}/{len(exr)}',
              status='PASS' if n_px == len(exr) == 10 else 'WARN',
              note=f'byte-identical (MD5) {n_md5}/{len(exr)}' + (f'; no raw file: {", ".join(missing)}' if missing else '')
              + (f'; pixels differ: {", ".join(differ)}' if differ else ''))])

print(f'\nPer-class instance / image counts of the dataset used from here on ({active["id"]}):')
print('\n'.join(class_table(active_counts)))
stored = []   # per-class instance counts stored in the repository's results/*.json (expected: none)
for f in sorted((repo_dir(REPO_TAG) / 'results').glob('*.json')):
    def _walk(o, path):
        if isinstance(o, dict):
            for k, v in o.items():
                if re.search(r'instance|count|support', str(k), re.I):
                    stored.append(f'{f.name}:{"/".join(path + [str(k)])}')
                _walk(v, path + [str(k)])
    _walk(jload(f), [])
summary_rows = [check_row(a['label'], 'all checks PASS', 'PASS' if a['ok'] else 'FAIL', ok=a['ok']) for a in attempts]
summary_rows.append(check_row('dataset used for Steps 6-8', 'a reproducing variant', active['id'], ok=split_ok))
summary_rows.append(check_row('per-class counts stored in results/*.json', 'compared if present',
                              ', '.join(stored[:5]) or 'none (the repository stores only metrics; Table 3.1 used)',
                              status='INFO'))
set_checks('5 summary', summary_rows)
write_diagnostics()
if split_ok:
    banner(f'PASS - the thesis split is reproduced by {active["id"]}')
else:
    banner('FAIL - no variant reproduced the thesis split. Steps 6-7 still run (diagnosis); Step 8 builds the zip '
           'only with FORCE. REPORT.txt and ui2compose_diagnostics.zip are on your Drive: send both back.', ok=False)

## Step 6 – Metric check with the released weights

`YOLO('runs/oversample_5k/weights/best.pt').val(data=<dataset.yaml>, split='test')` and `split='val'` are run with the Ultralytics validation settings that produced the published numbers: `imgsz=640`, `batch=16` (= training-time validation batch 8×2), `conf=0.001`, `iou=0.7`, `max_det=300`, `rect=True`, FP32. On a GPU the run is repeated in FP16, because the published **val** value (51.82 %) is the training-time validation value, and training ran with AMP (FP16).

The results are compared with three references:
* test: `results/perclass_ap_results.json["oversample_5k"]`: mAP@50 0.48699, mAP@50-95 0.39109, and AP@50 for each of the 13 classes;
* val: `results/oversample_5k_metrics.json`: mAP@50 0.51823, mAP@50-95 0.40173;
* the thesis re-evaluation of the same weights outside the training environment (**Table A.2**): val mAP@50 51.87 %, test mAP@50 48.74 %.

A split passes if, for FP32 or FP16, mAP@50 and mAP@50-95 are within `MAP_TOL` = 0.005 (0.5 points) of the repository values, every per-class AP@50 on test is within `CLASS_TOL` = 0.02, **and** mAP@50 is within `REEVAL_TOL` = 0.0015 (0.15 points) of Table A.2. All differences are printed. A wrong split shows up as differences of several points. A metric FAIL is reported but does not block the zip.

Runtime: about 5–10 min in total on a T4 GPU, about **40–55 min on a CPU runtime** (2 vCPUs, ~0.8 s per image, 3,739 images). Results are saved as JSON, so a restart does not repeat them.

In [ ]:
#@title Step 6 - Evaluate best.pt on val and test, compare with the published metrics
act = load_state('active')
if not act:
    raise PipelineStop('Step 5 has not finished - run it first')
clone = repo_dir(REPO_TAG)
weights = clone / 'runs' / 'oversample_5k' / 'weights' / 'best.pt'
pub_test = jload(clone / 'results' / 'perclass_ap_results.json')['oversample_5k']
pub_val = jload(clone / 'results' / 'oversample_5k_metrics.json')
published = {
    'test': {'mAP50': pub_test['overall']['mAP50'], 'mAP50_95': pub_test['overall']['mAP50_95'],
             'precision': pub_test['overall'].get('precision'), 'recall': pub_test['overall'].get('recall'),
             'per_class_ap50': pub_test['per_class_ap50'], 'source': 'results/perclass_ap_results.json[oversample_5k]'},
    'val': {'mAP50': pub_val['mAP50'], 'mAP50_95': pub_val['mAP50_95'], 'per_class_ap50': {},
            'precision': pub_val.get('precision'), 'recall': pub_val.get('recall'),
            'source': 'results/oversample_5k_metrics.json (training-time validation, epoch %s)' % pub_val.get('epoch')}}

import torch
gpu = torch.cuda.is_available()
device = 0 if gpu else 'cpu'
variants = [('fp32', False)] + ([('fp16', True)] if gpu and ALSO_RUN_FP16_ON_GPU else [])
n_img = {s: act['counts'][s]['images'] for s in ('val', 'test')}
per_img = 0.1 if gpu else 0.8   # seconds per image (Colab: T4 / 2 vCPUs)
print(f'device: {"GPU " + torch.cuda.get_device_name(0) if gpu else "CPU"}; variants: {[v for v, _ in variants]}; '
      f'expected time ~{sum(n_img.values()) * per_img * len(variants) / 60:.0f} min')


def run_val(split, half):
    from ultralytics import YOLO
    import ultralytics
    t0 = time.time()
    model = YOLO(str(weights))
    r = model.val(data=str(act['yaml']), split=split, half=half, device=device, plots=False, save_json=False,
                  verbose=True, project=str(D['metrics'] / 'ultralytics_runs'),
                  name=f'{act["id"]}_{split}_{"fp16" if half else "fp32"}', exist_ok=True, **VAL_SETTINGS)
    box, names = r.box, r.names
    per_class = {names[int(c)]: {'ap50': float(box.ap50[k]), 'ap50_95': float(box.ap[k])}
                 for k, c in enumerate(box.ap_class_index)}
    nt = getattr(r, 'nt_per_class', None)
    return {'dataset': act['id'], 'split': split, 'variant': 'fp16' if half else 'fp32', 'images': n_img[split],
            'mAP50': float(box.map50), 'mAP50_95': float(box.map), 'precision': float(box.mp), 'recall': float(box.mr),
            'per_class': per_class,
            'instances_per_class': {names[i]: int(n) for i, n in enumerate(nt)} if nt is not None else None,
            'speed_ms_per_image': {k: float(v) for k, v in dict(r.speed).items()}, 'seconds': round(time.time() - t0),
            'settings': {**VAL_SETTINGS, 'half': half, 'device': str(device), 'data': str(act['yaml']),
                         'weights': str(weights), 'weights_sha256': file_hash(weights)},
            'versions': {'ultralytics': ultralytics.__version__, 'torch': torch.__version__, 'cuda': torch.version.cuda,
                         'gpu': torch.cuda.get_device_name(0) if gpu else None},
            'published': published[split]}


results = {}
for split in ('test', 'val'):
    for vname, half in variants:
        key = f'metrics_{act["id"]}_{split}_{vname}'
        path = D['metrics'] / f'metrics_{act["id"]}_{split}_{vname}.json'
        if done(key) and path.exists():
            results[(split, vname)] = jload(path)
            print(f'[skip] {split} {vname} already evaluated: mAP50 {results[(split, vname)]["mAP50"]:.5f}')
            continue
        print(f'\n--- YOLO.val split={split} {vname} on {n_img[split]:,} images ---', flush=True)
        m = run_val(split, half)
        jdump(m, path)
        mark(key, {'file': str(path), 'mAP50': m['mAP50'], 'mAP50_95': m['mAP50_95']})
        results[(split, vname)] = m

rows, split_pass = [], {}
for split in ('test', 'val'):
    pub, passing = published[split], []
    for vname, _ in variants:
        m = results[(split, vname)]
        d50, d95 = m['mAP50'] - pub['mAP50'], m['mAP50_95'] - pub['mAP50_95']
        vr = [check_row(f'{split} mAP@50 [{vname}]', pub['mAP50'], m['mAP50'], ok=abs(d50) <= MAP_TOL,
                        note=f'diff {100 * d50:+.2f} points'),
              check_row(f'{split} mAP@50-95 [{vname}]', pub['mAP50_95'], m['mAP50_95'], ok=abs(d95) <= MAP_TOL,
                        note=f'diff {100 * d95:+.2f} points')]
        reev = EXPECTED[f'reeval_{split}_mAP50']   # Table A.2: re-evaluated outside the training environment
        vr.append(check_row(f'{split} mAP@50 = thesis re-evaluation (Table A.2) [{vname}]', reev, m['mAP50'],
                            ok=abs(m['mAP50'] - reev) <= REEVAL_TOL,
                            note=f'diff {100 * (m["mAP50"] - reev):+.2f} points, tolerance {100 * REEVAL_TOL:.2f}'))
        if pub['per_class_ap50']:
            diffs = {n: (m['per_class'].get(n, {}).get('ap50') or 0.0) - v for n, v in pub['per_class_ap50'].items()}
            bad = [f'{n} {100 * d:+.1f}' for n, d in diffs.items() if abs(d) > CLASS_TOL]
            vr.append(check_row(f'{split} per-class AP@50 within {CLASS_TOL} [{vname}]', f'{len(diffs)}/{len(diffs)}',
                                f'{len(diffs) - len(bad)}/{len(diffs)}', ok=not bad,
                                note=('outside: ' + ', '.join(bad)) if bad else
                                f'max |diff| {100 * max(map(abs, diffs.values())):.2f} points'))
        if all(r['status'] == 'PASS' for r in vr):
            passing.append(vname)
        for k in ('precision', 'recall'):  # informational: depend on the confidence threshold of the P/R point
            if pub.get(k) is not None:
                vr.append(check_row(f'{split} {k} [{vname}]', pub[k], m[k], status='INFO',
                                    note=f'diff {100 * (m[k] - pub[k]):+.2f} points'))
        rows += vr
        if m.get('instances_per_class'):
            ours = act['counts'][split]['per_class_instances']
            same = all(m['instances_per_class'].get(k, 0) == v for k, v in ours.items())
            rows.append(check_row(f'{split} Ultralytics instance counts = label lines [{vname}]', True, same,
                                  status='PASS' if same else 'WARN'))
    split_pass[split] = bool(passing)
    rows.append(check_row(f'{split} metrics reproduced (FP32 or FP16)', True, bool(passing),
                          note=('by ' + ', '.join(passing)) if passing else ''))
metrics_ok = all(split_pass.values())
set_checks('6 metrics', rows)
save_state('metrics_summary', {'ok': metrics_ok, 'per_split': split_pass, 'dataset': act['id'],
                               'values': {f'{s} {v}': {'mAP50': m['mAP50'], 'mAP50_95': m['mAP50_95']}
                                          for (s, v), m in results.items()},
                               'files': sorted(str(p) for p in D['metrics'].glob(f'metrics_{act["id"]}_*.json'))})
write_diagnostics()
banner(('PASS' if metrics_ok else 'FAIL') + f' - published metrics {"reproduced" if metrics_ok else "NOT reproduced"} '
       f'on {act["id"]} (decisive rows: "... metrics reproduced")', ok=metrics_ok)

## Step 7 – Hash manifest of all screens with `app_split` (v1.2.0), unseen-app lists

The next cell writes `src/ui2compose/preprocess/app_split.py` from ui2compose v1.2.0, which is not on GitHub yet. The cell is long and you don't need to read it. The cell after it checks that the written file is byte-identical to the reviewed version (SHA-256 `60b693d1719f5241cf2f0bb06a4e229e8c6c44770ff7987f4fcf10736f2b388b`). It then runs the tool on the YOLO dataset (all three splits):
* **full grouping** (default): package OR identical MD5 OR dHash distance ≤ 4. This writes `manifest.csv` (every screen with package, group, split, MD5, dHash, exclusion reason and matched training screen), `unseen_val.txt`, `unseen_test.txt` and `summary.json` with `split_fingerprints`. It hashes all 18,694 PNGs: about 12–16 min on Colab's 2 CPUs.
* **package + MD5** (`--near-dup-dist -1`), for comparison. This runs the same `main()` of the tool, but takes the MD5 and dHash of every screen from the first manifest instead of decoding all PNGs again (about 1 min).

The **package-only** numbers of the thesis (214 unseen test screens; 1,595 test apps, 1,392 of them also in train) are computed from the manifest. They must equal Step 5, which already blocks the zip if they differ.

**Pixel-identical screens** (Table A.2: 9 val, 8 test, "ReDraw white-label clones"): every val/test screen whose dHash equals that of a training screen is compared with it, first by MD5 and, if the bytes differ, by decoded pixels. The table also shows the byte-identical (MD5) count and how many matches come from another package. A difference is reported as WARN, because the thesis does not say exactly how it counted, and it does not block the zip. The matched pairs are saved in `splits/pixel_identical_to_train.csv`.

In [ ]:
%%writefile ui2compose_app_split_v1.2.0.py
# SPDX-FileCopyrightText: 2026 Murat Saran <saran@cankaya.edu.tr>
# SPDX-FileCopyrightText: 2026 Göktürk Bilgetürk <gbilgeturk@yahoo.com>
#
# SPDX-License-Identifier: MIT

"""App-level grouping of screens: leak audit, unseen-app subset, app-level split.

The v1.1.0 dataset was split at screenshot level (build_dataset.py shuffles
individual screens), so screens of the same app, and pixel-identical copies of
a screen published under another package name, can occur in train and in
val/test. This module makes that measurable and avoidable without retraining:

  * Every screen gets a *group*: the connected component (union-find) of the
    relation "same package OR identical file content (MD5) OR near-identical
    image (dHash Hamming distance <= k)", optionally also "same app-builder
    family" (see builder_family()). The relation is applied transitively.
  * unseen_subset() keeps the evaluation screens whose group contains no
    training screen (the "unseen-app subset") and reports, for every excluded
    screen, the first reason that applies (package, builder family, MD5,
    near-duplicate, or only transitively through other screens of its group).
  * split_by_group() makes a new group-level split (for future retraining)
    with greedy multi-label stratification over per-group class counts, and
    audit_split() checks a split for leakage independently of the grouping.
  * split_fingerprints() records a SHA-256 of every split's sorted stem list
    (summary.json), so that a regenerated dataset can be compared with the one
    the released weights were evaluated on (--expect-fingerprints).

The unseen-app subset is only meaningful if the dataset given here has
exactly the split the released weights were trained on. This module cannot
check that by itself (the original split lists were not archived with
v1.1.0); evaluate_detector.py --expect-metrics checks the necessary condition
that the released weights reproduce the published val and test metrics on
the regenerated split.

File naming. build_dataset.py names every dataset image
"<package>-screens_screenshot_<n>.<ext>" (ReDraw app folder "<package>-screens"
+ "_" + "screenshot_<n>"); oversample_dataset.py appends "_os<k>" to the stem
of every copy it writes (repeated runs would append it again). parse_stem()
is anchored on exactly this pattern and never splits on '_' or '.' blindly,
because package names contain both.

Near-duplicate search scales to the full ReDraw set (~19k screens): the 64-bit
dHash is split into k+1 disjoint bands; by the pigeonhole principle two hashes
within Hamming distance k agree exactly on at least one band, so only pairs
sharing a band value are candidates (multi-index hashing), and their exact
distance is then verified with vectorised NumPy popcounts.

Dependencies: standard library, NumPy and Pillow only.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import os
import re
import sys
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path
from typing import (Dict, Iterable, List, Mapping, NamedTuple, Optional,
                    Sequence, Tuple, Union)

import numpy as np
from PIL import Image

# ---------------------------------------------------------------------------
# Constants and parameters
# ---------------------------------------------------------------------------

SPLITS: Tuple[str, str, str] = ("train", "val", "test")
IMG_EXTS: Tuple[str, ...] = (".png", ".jpg", ".jpeg")

# Maximum dHash Hamming distance (out of 64 bits) at which two screens are
# treated as near-identical. PROVISIONAL default: chosen a priori (a common
# choice for 64-bit difference hashes), not tuned. It should be checked on the
# data by inspecting the near-duplicate matches written to the manifest
# (columns exclusion_reason / matched_train_stem / match_distance); it must
# not be tuned on test-set metrics.
DEFAULT_NEAR_DUP_MAX_DIST = 4

# dHash grid size: a (size+1) x size grayscale thumbnail gives size*size bits.
DEFAULT_DHASH_SIZE = 8

# Default split ratios and seed (identical to build_dataset.py: SPLIT, SEED).
DEFAULT_RATIOS: Tuple[float, float, float] = (0.80, 0.10, 0.10)
DEFAULT_SEED = 42

# Exclusion reasons reported by unseen_subset(), in the order they are tested.
REASON_PACKAGE = "package"
REASON_BUILDER = "builder_prefix"
REASON_MD5 = "md5"
REASON_NEAR_DUP = "near_dup"
REASON_TRANSITIVE = "transitive"
EXCLUSION_REASONS: Tuple[str, ...] = (REASON_PACKAGE, REASON_BUILDER, REASON_MD5,
                                      REASON_NEAR_DUP, REASON_TRANSITIVE)

# "<package>-screens_screenshot_<n>" followed by zero or more "_os<k>" copy
# suffixes. The package part is greedy so that the anchor is the LAST
# "-screens_screenshot_<n>" before the copy suffixes.
_STEM_RE = re.compile(
    r"^(?P<package>.+)-screens_screenshot_(?P<index>\d+)(?P<copies>(?:_os\d+)*)$")
_COPY_RE = re.compile(r"_os(\d+)")

# A package segment that looks like an app-builder generated id: letters
# followed by at least three digits (e.g. 'dev231870' in
# 'com.andromo.dev231870.app482676').
_BUILDER_SEGMENT_RE = re.compile(r"^(?P<prefix>[A-Za-z]+)(?P<digits>\d{3,})$")


class ParsedStem(NamedTuple):
    """Components of a dataset image stem.

    package      — Android package name (as written in the file name)
    screen_index — the <n> of "screenshot_<n>"
    copy_indices — the <k> of every "_os<k>" oversampling suffix, in order
                   (empty tuple for an original screen)
    base_stem    — the stem without oversampling suffixes
    """
    package: str
    screen_index: int
    copy_indices: Tuple[int, ...]
    base_stem: str


@dataclass
class ScreenRecord:
    """One dataset image and the attributes used for grouping.

    stem         — file stem, e.g. "com.shazam.android-screens_screenshot_3"
    path         — image path (None for records built in memory)
    split        — "train" / "val" / "test" (or None)
    package      — parsed package name (see parse_package)
    md5          — hex MD5 of the image file (None: not computed)
    dhash        — dHash as an int (None: not computed)
    class_counts — class id -> number of label instances in this image
    is_copy      — True for an oversampled copy ("_os<k>" suffix)
    """
    stem: str
    path: Optional[str] = None
    split: Optional[str] = None
    package: Optional[str] = None
    md5: Optional[str] = None
    dhash: Optional[int] = None
    class_counts: Dict[int, int] = field(default_factory=dict)
    is_copy: bool = False


class SplitLeakageError(AssertionError):
    """Raised by audit_split() when a split leaks screens across subsets."""


# ---------------------------------------------------------------------------
# File names
# ---------------------------------------------------------------------------

def parse_stem(stem: str) -> ParsedStem:
    """Parses a dataset image stem into package, screen index and copy suffixes.

    Input:  stem — file name without extension, e.g.
            "com.shazam.android-screens_screenshot_3" or
            "com.shazam.android-screens_screenshot_3_os0"
    Output: ParsedStem(package, screen_index, copy_indices, base_stem);
            raises ValueError if the stem does not follow the
            "<package>-screens_screenshot_<n>[_os<k>...]" pattern
    """
    m = _STEM_RE.match(stem)
    if not m:
        raise ValueError(
            f"stem {stem!r} does not match '<package>-screens_screenshot_<n>"
            f"[_os<k>...]' (the naming produced by build_dataset.py / "
            f"oversample_dataset.py)")
    copies = tuple(int(k) for k in _COPY_RE.findall(m.group("copies")))
    base = stem[: len(stem) - len(m.group("copies"))]
    return ParsedStem(m.group("package"), int(m.group("index")), copies, base)


def parse_package(stem: str) -> str:
    """Returns the Android package name encoded in a dataset image stem.

    Input:  stem — file name without extension (see parse_stem)
    Output: package name, e.g. "com.shazam.android"; raises ValueError for a
            stem that does not follow the dataset naming pattern
    """
    return parse_stem(stem).package


def builder_family(package: str) -> Optional[str]:
    """Returns the app-builder family key of a package, if it has one.

    The key is the package up to and including the letters of the first
    segment that consists of letters followed by >= 3 digits, e.g.
    'com.andromo.dev231870.app482676' -> 'com.andromo.dev'. Apps generated by
    the same builder account share it. This is a heuristic and is only used
    when builder-prefix grouping is switched on.

    Input:  package — Android package name
    Output: family key (lower case), or None if no segment matches
    """
    parts = package.lower().split(".")
    for i, seg in enumerate(parts):
        m = _BUILDER_SEGMENT_RE.match(seg)
        if m and i > 0:
            return ".".join(parts[:i] + [m.group("prefix")])
    return None


# ---------------------------------------------------------------------------
# Hashing
# ---------------------------------------------------------------------------

def md5_file(path: Union[str, Path], chunk_size: int = 1 << 20) -> str:
    """Computes the MD5 digest of a file's bytes.

    Input:  path — file path; chunk_size — read size in bytes
    Output: lower-case hexadecimal MD5 string
    """
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def dhash(image: Union[str, Path, Image.Image], size: int = DEFAULT_DHASH_SIZE) -> int:
    """Computes the difference hash (dHash) of an image.

    The image is converted to 8-bit grayscale, resized to (size+1) x size
    pixels (width x height, Lanczos filter), and bit (r, c) is 1 when pixel
    (r, c+1) is brighter than its left neighbour (r, c). Bits are packed row
    by row, most significant bit first.

    Input:  image — image path or PIL image; size — grid size (bits = size*size)
    Output: hash as a non-negative int with size*size bits
    """
    if size < 1:
        raise ValueError("size must be >= 1")
    if isinstance(image, Image.Image):
        img = image
        gray = img.convert("L").resize((size + 1, size), Image.Resampling.LANCZOS)
    else:
        with Image.open(image) as img:
            gray = img.convert("L").resize((size + 1, size), Image.Resampling.LANCZOS)
    px = np.asarray(gray, dtype=np.int16)
    bits = (px[:, 1:] > px[:, :-1]).ravel()
    value = 0
    for b in bits:
        value = (value << 1) | int(b)
    return value


def hamming(a: int, b: int) -> int:
    """Counts the differing bits of two integer hashes.

    Input:  a, b — non-negative ints
    Output: Hamming distance (number of set bits of a XOR b)
    """
    return bin(a ^ b).count("1")


_POPCOUNT_TABLE = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def _popcount_u64(x: np.ndarray) -> np.ndarray:
    """Counts set bits of every element of a uint64 array.

    Input:  x — np.ndarray of dtype uint64
    Output: np.ndarray of the same shape with the bit counts (int64)
    """
    x = np.ascontiguousarray(x, dtype=np.uint64)
    if hasattr(np, "bitwise_count"):  # NumPy >= 2.0
        return np.bitwise_count(x).astype(np.int64)
    as_bytes = x.view(np.uint8).reshape(x.shape + (8,))
    return _POPCOUNT_TABLE[as_bytes].sum(axis=-1).astype(np.int64)


def _bands(nbits: int, max_dist: int) -> List[Tuple[int, int]]:
    """Splits nbits into max_dist+1 disjoint contiguous bands.

    Input:  nbits — hash length in bits; max_dist — Hamming radius k
    Output: list of (shift, width) per band; widths differ by at most one
    """
    n_bands = max_dist + 1
    base, extra = divmod(nbits, n_bands)
    bands, shift = [], 0
    for i in range(n_bands):
        width = base + (1 if i < extra else 0)
        bands.append((shift, width))
        shift += width
    return bands


class HammingIndex:
    """Multi-index hashing over a set of <= 64-bit hashes for radius-k search."""

    def __init__(self, hashes: Sequence[int], max_dist: int, nbits: int = 64):
        """Builds one exact-match table per band.

        Input:  hashes — hash values (ints < 2**nbits); max_dist — radius k
                (0 <= k < nbits); nbits — hash length (<= 64)
        Output: none (index structures stored on the instance)
        """
        if not 1 <= nbits <= 64:
            raise ValueError("nbits must be in [1, 64]")
        if not 0 <= max_dist < nbits:
            raise ValueError(f"max_dist must be in [0, {nbits - 1}]")
        self.max_dist = max_dist
        self.nbits = nbits
        self.hashes = np.array([int(h) for h in hashes], dtype=np.uint64)
        self.bands = _bands(nbits, max_dist)
        self.tables: List[Dict[int, np.ndarray]] = []
        for shift, width in self.bands:
            vals = (self.hashes >> np.uint64(shift)) & np.uint64((1 << width) - 1)
            order = np.argsort(vals, kind="stable")
            sorted_vals = vals[order]
            cuts = np.flatnonzero(np.diff(sorted_vals)) + 1
            table: Dict[int, np.ndarray] = {}
            for run in np.split(order, cuts):
                if run.size:
                    table[int(vals[run[0]])] = run
            self.tables.append(table)

    def query(self, h: int, max_dist: Optional[int] = None) -> List[Tuple[int, int]]:
        """Finds all indexed hashes within Hamming distance of h.

        Input:  h — query hash; max_dist — radius (<= the index radius;
                default the index radius)
        Output: sorted list of (index, distance) pairs
        """
        k = self.max_dist if max_dist is None else max_dist
        if k > self.max_dist:
            raise ValueError("query radius exceeds the index radius")
        cand: List[np.ndarray] = []
        for (shift, width), table in zip(self.bands, self.tables):
            run = table.get((int(h) >> shift) & ((1 << width) - 1))
            if run is not None:
                cand.append(run)
        if not cand:
            return []
        idx = np.unique(np.concatenate(cand))
        d = _popcount_u64(self.hashes[idx] ^ np.uint64(int(h)))
        keep = d <= k
        return [(int(i), int(di)) for i, di in zip(idx[keep], d[keep])]

    def pairs(self) -> List[Tuple[int, int, int]]:
        """Finds all index pairs within the index radius.

        Input:  none
        Output: sorted list of (i, j, distance) with i < j
        """
        found: Dict[Tuple[int, int], int] = {}
        for table in self.tables:
            for run in table.values():
                if run.size < 2:
                    continue
                run = np.sort(run)
                hs = self.hashes[run]
                for p in range(run.size - 1):
                    d = _popcount_u64(hs[p + 1:] ^ hs[p])
                    for q in np.flatnonzero(d <= self.max_dist):
                        found[(int(run[p]), int(run[p + 1 + q]))] = int(d[q])
        return sorted((i, j, d) for (i, j), d in found.items())


def near_duplicate_pairs(hashes: Sequence[int], max_dist: int,
                         nbits: int = 64) -> List[Tuple[int, int, int]]:
    """Finds all pairs of hashes within a Hamming radius (multi-index hashing).

    Input:  hashes — hash values; max_dist — radius k; nbits — hash length
    Output: sorted list of (i, j, distance) with i < j (indices into hashes)
    """
    return HammingIndex(hashes, max_dist, nbits).pairs()


# ---------------------------------------------------------------------------
# Grouping
# ---------------------------------------------------------------------------

class UnionFind:
    """Disjoint-set forest with path halving and union by size."""

    def __init__(self, n: int):
        """Creates n singleton sets.

        Input:  n — number of elements
        Output: none
        """
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, a: int) -> int:
        """Returns the representative of a's set.

        Input:  a — element index
        Output: representative index
        """
        while self.parent[a] != a:
            self.parent[a] = self.parent[self.parent[a]]
            a = self.parent[a]
        return a

    def union(self, a: int, b: int) -> bool:
        """Merges the sets of a and b.

        Input:  a, b — element indices
        Output: True if two different sets were merged
        """
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True


def _package_key(rec: ScreenRecord) -> Optional[str]:
    """Returns the package grouping key of a record (lower case).

    Input:  rec — screen record
    Output: lower-cased package, or None if unknown
    """
    return rec.package.lower() if rec.package else None


def _hash_bits(records: Sequence[ScreenRecord]) -> int:
    """Returns the bit length used for near-duplicate search (always 64).

    Shorter hashes (dhash(size < 8)) are handled as 64-bit values whose high
    bits are zero, which keeps the multi-index search exact.

    Input:  records — screen records
    Output: 64; raises ValueError if a hash does not fit in 64 bits
    """
    max_h = max((r.dhash for r in records if r.dhash is not None), default=0)
    if max_h >= 1 << 64:
        raise ValueError("near-duplicate grouping supports hashes of at most 64 bits")
    return 64


def find_links(records: Sequence[ScreenRecord],
               near_dup_max_dist: Optional[int] = DEFAULT_NEAR_DUP_MAX_DIST,
               builder_prefix: bool = False) -> Dict[str, List[Tuple[int, int, int]]]:
    """Lists the pairwise links that define the screen groups.

    Package, builder-family and MD5 links are emitted as chains (each record is
    linked to the first record with the same key), which is sufficient for
    union-find. Near-duplicate links are computed between distinct dHash
    values with multi-index hashing; records with an identical dHash are
    chained with distance 0.

    Input:  records — screen records (package / md5 / dhash may be None);
            near_dup_max_dist — dHash radius k (None disables near-dup links);
            builder_prefix — also link records of the same builder family
    Output: {'package': [...], 'builder_prefix': [...], 'md5': [...],
             'near_dup': [...]} — lists of (i, j, distance) record-index
            triples (distance is 0 except for near-dup links)
    """
    links: Dict[str, List[Tuple[int, int, int]]] = {
        REASON_PACKAGE: [], REASON_BUILDER: [], REASON_MD5: [], REASON_NEAR_DUP: []}

    def chain(key_fn, name: str) -> None:
        """Links every record to the first record sharing its key.

        Input:  key_fn — record -> key (None: no link); name — link type
        Output: none (appends to links[name])
        """
        first: Dict[object, int] = {}
        for i, rec in enumerate(records):
            key = key_fn(rec)
            if key is None:
                continue
            if key in first:
                links[name].append((first[key], i, 0))
            else:
                first[key] = i

    chain(_package_key, REASON_PACKAGE)
    if builder_prefix:
        chain(lambda r: builder_family(r.package) if r.package else None, REASON_BUILDER)
    chain(lambda r: r.md5, REASON_MD5)

    if near_dup_max_dist is not None:
        by_hash: Dict[int, List[int]] = defaultdict(list)
        for i, rec in enumerate(records):
            if rec.dhash is not None:
                by_hash[int(rec.dhash)].append(i)
        if by_hash:
            nbits = _hash_bits(records)
            uniq = sorted(by_hash)
            for h in uniq:
                members = by_hash[h]
                links[REASON_NEAR_DUP].extend((members[0], m, 0) for m in members[1:])
            for a, b, d in near_duplicate_pairs(uniq, near_dup_max_dist, nbits):
                links[REASON_NEAR_DUP].append((by_hash[uniq[a]][0], by_hash[uniq[b]][0], d))
    return links


def group_screens(records: Sequence[ScreenRecord],
                  near_dup_max_dist: Optional[int] = DEFAULT_NEAR_DUP_MAX_DIST,
                  builder_prefix: bool = False) -> List[int]:
    """Assigns every screen to a leak group (connected component of the links).

    Two screens are in the same group if they are connected by a chain of
    links "same package (case-insensitive)", "identical MD5", "dHash distance
    <= near_dup_max_dist" and, if builder_prefix, "same builder family".
    Group ids are canonical: groups are numbered 0, 1, ... in the order of
    their lexicographically smallest stem, so they do not depend on the order
    of the input records.

    Input:  records — screen records; near_dup_max_dist — dHash radius k
            (None disables near-dup links); builder_prefix — enable
            builder-family links (see builder_family)
    Output: list of group ids aligned with records
    """
    uf = UnionFind(len(records))
    for pairs in find_links(records, near_dup_max_dist, builder_prefix).values():
        for i, j, _ in pairs:
            uf.union(i, j)
    root_min_stem: Dict[int, str] = {}
    for i, rec in enumerate(records):
        r = uf.find(i)
        if r not in root_min_stem or rec.stem < root_min_stem[r]:
            root_min_stem[r] = rec.stem
    order = sorted(root_min_stem, key=lambda r: root_min_stem[r])
    gid = {r: g for g, r in enumerate(order)}
    return [gid[uf.find(i)] for i in range(len(records))]


# ---------------------------------------------------------------------------
# Unseen-app subset
# ---------------------------------------------------------------------------

@dataclass
class Exclusion:
    """Why an evaluation screen is not in the unseen subset.

    reason        — one of EXCLUSION_REASONS
    matched_stem  — a training stem that triggered the reason
    distance      — dHash distance for 'near_dup' (0 for 'md5'), else None
    """
    reason: str
    matched_stem: Optional[str] = None
    distance: Optional[int] = None


def classify_exclusions(train_records: Sequence[ScreenRecord],
                        eval_records: Sequence[ScreenRecord],
                        near_dup_max_dist: Optional[int] = DEFAULT_NEAR_DUP_MAX_DIST,
                        builder_prefix: bool = False
                        ) -> Tuple[List[Optional[Exclusion]], List[int]]:
    """Decides, for every evaluation screen, whether its group touches training.

    Reasons are tested in this order and the first that applies is reported:
    'package' (same package as a training screen), 'builder_prefix' (same
    builder family; only if enabled), 'md5' (byte-identical to a training
    image), 'near_dup' (dHash distance <= k to a training image; the closest
    one is reported), 'transitive' (none of the above, but the group reaches a
    training screen through other screens, e.g. a near-duplicate of a test
    screen whose package is in training).

    Input:  train_records, eval_records — screen records;
            near_dup_max_dist — dHash radius k (None disables);
            builder_prefix — enable builder-family grouping
    Output: (exclusions, group_ids) — exclusions aligned with eval_records
            (None = unseen); group_ids aligned with train_records + eval_records
    """
    all_records = list(train_records) + list(eval_records)
    n_train = len(train_records)
    gids = group_screens(all_records, near_dup_max_dist, builder_prefix)
    train_groups = set(gids[:n_train])

    first_by_pkg: Dict[str, str] = {}
    first_by_family: Dict[str, str] = {}
    first_by_md5: Dict[str, str] = {}
    first_by_group: Dict[int, str] = {}
    for rec, g in sorted(zip(train_records, gids[:n_train]), key=lambda t: t[0].stem):
        key = _package_key(rec)
        if key is not None:
            first_by_pkg.setdefault(key, rec.stem)
            fam = builder_family(rec.package) if builder_prefix else None
            if fam is not None:
                first_by_family.setdefault(fam, rec.stem)
        if rec.md5 is not None:
            first_by_md5.setdefault(rec.md5, rec.stem)
        first_by_group.setdefault(g, rec.stem)

    # Near-duplicate index over the distinct training hashes; each hash maps to
    # its lexicographically first training stem.
    index: Optional[HammingIndex] = None
    stem_of_hash: Dict[int, str] = {}
    for rec in train_records:
        if rec.dhash is not None:
            h = int(rec.dhash)
            stem_of_hash[h] = min(stem_of_hash.get(h, rec.stem), rec.stem)
    train_hashes = sorted(stem_of_hash)
    if near_dup_max_dist is not None and train_hashes:
        index = HammingIndex(train_hashes, near_dup_max_dist, _hash_bits(all_records))

    exclusions: List[Optional[Exclusion]] = []
    for rec, g in zip(eval_records, gids[n_train:]):
        if g not in train_groups:
            exclusions.append(None)
            continue
        key = _package_key(rec)
        fam = builder_family(rec.package) if (builder_prefix and rec.package) else None
        if key is not None and key in first_by_pkg:
            exclusions.append(Exclusion(REASON_PACKAGE, first_by_pkg[key]))
        elif fam is not None and fam in first_by_family:
            exclusions.append(Exclusion(REASON_BUILDER, first_by_family[fam]))
        elif rec.md5 is not None and rec.md5 in first_by_md5:
            exclusions.append(Exclusion(REASON_MD5, first_by_md5[rec.md5], 0))
        else:
            hits = index.query(rec.dhash) if (index is not None and rec.dhash is not None) else []
            if hits:
                best = min(hits, key=lambda t: (t[1], stem_of_hash[train_hashes[t[0]]]))
                exclusions.append(Exclusion(REASON_NEAR_DUP,
                                            stem_of_hash[train_hashes[best[0]]], best[1]))
            else:
                exclusions.append(Exclusion(REASON_TRANSITIVE, first_by_group[g]))
    return exclusions, gids


def unseen_subset(train_records: Sequence[ScreenRecord],
                  eval_records: Sequence[ScreenRecord],
                  near_dup_max_dist: Optional[int] = DEFAULT_NEAR_DUP_MAX_DIST,
                  builder_prefix: bool = False
                  ) -> Tuple[List[ScreenRecord], Dict[str, object]]:
    """Selects the evaluation screens whose leak group contains no training screen.

    Input:  train_records — training screens (oversampled copies may be
            included; they share package and MD5 with their originals);
            eval_records — validation or test screens;
            near_dup_max_dist — dHash radius k (None disables near-dup links);
            builder_prefix — enable builder-family grouping
    Output: (subset, stats) — subset in input order; stats = {'n_eval',
            'n_unseen', 'n_excluded', 'excluded_by_reason': {reason: count}}
    """
    exclusions, _ = classify_exclusions(train_records, eval_records,
                                        near_dup_max_dist, builder_prefix)
    subset = [rec for rec, ex in zip(eval_records, exclusions) if ex is None]
    by_reason = Counter(ex.reason for ex in exclusions if ex is not None)
    stats: Dict[str, object] = {
        "n_eval": len(eval_records),
        "n_unseen": len(subset),
        "n_excluded": len(eval_records) - len(subset),
        "excluded_by_reason": {r: by_reason.get(r, 0) for r in EXCLUSION_REASONS},
    }
    return subset, stats


# ---------------------------------------------------------------------------
# Group-level split
# ---------------------------------------------------------------------------

def _check_ratios(ratios: Sequence[float]) -> np.ndarray:
    """Validates split ratios.

    Input:  ratios — non-negative fractions summing to 1
    Output: ratios as a float64 array; raises ValueError otherwise
    """
    r = np.asarray(ratios, dtype=np.float64)
    if r.ndim != 1 or r.size < 2 or (r < 0).any() or abs(r.sum() - 1.0) > 1e-6:
        raise ValueError(f"ratios must be >= 0 and sum to 1, got {list(ratios)}")
    return r


def split_by_group(records: Sequence[ScreenRecord],
                   ratios: Sequence[float] = DEFAULT_RATIOS,
                   seed: int = DEFAULT_SEED,
                   stratify: bool = True,
                   group_ids: Optional[Sequence[int]] = None,
                   near_dup_max_dist: Optional[int] = DEFAULT_NEAR_DUP_MAX_DIST,
                   builder_prefix: bool = False,
                   split_names: Sequence[str] = SPLITS) -> List[str]:
    """Splits screens so that every leak group lands in exactly one subset.

    Stratified mode (greedy multi-label iterative stratification, after
    Sechidis et al. 2011, applied to groups instead of single examples):
    the desired number of instances of every class in every subset is
    ratio * total. Repeatedly, the class with the fewest instances left in
    unassigned groups is taken; its groups are visited in descending order of
    their instance count of that class (ties in a seeded random order), and
    each group goes to the subset with the largest remaining demand for that
    class (ties: largest remaining demand in screens, then seeded random).
    Groups without labels are finally assigned by remaining screen demand.
    Unstratified mode visits all groups in a seeded random order and assigns
    by remaining screen demand only.

    The result depends only on the records' content and the seed, not on the
    order of the records.

    Input:  records — screen records with class_counts; ratios — subset
            fractions (sum 1); seed — RNG seed for tie-breaking/order;
            stratify — use the class-aware greedy; group_ids — precomputed
            group ids (default: group_screens(records, near_dup_max_dist,
            builder_prefix)); split_names — subset names aligned with ratios
    Output: list of subset names aligned with records
    """
    r = _check_ratios(ratios)
    if len(split_names) != r.size:
        raise ValueError("split_names and ratios must have the same length")
    if group_ids is None:
        group_ids = group_screens(records, near_dup_max_dist, builder_prefix)
    if len(group_ids) != len(records):
        raise ValueError("group_ids must align with records")

    # Canonical group order (by smallest stem) so the input order does not matter.
    members: Dict[int, List[int]] = defaultdict(list)
    for i, g in enumerate(group_ids):
        members[g].append(i)
    groups = sorted(members, key=lambda g: min(records[i].stem for i in members[g]))
    classes = sorted({c for rec in records for c in rec.class_counts})
    cidx = {c: k for k, c in enumerate(classes)}
    G = np.zeros((len(groups), len(classes)), dtype=np.float64)
    size = np.zeros(len(groups), dtype=np.float64)
    for gi, g in enumerate(groups):
        for i in members[g]:
            size[gi] += 1
            for c, n in records[i].class_counts.items():
                G[gi, cidx[c]] += n

    rng = np.random.default_rng(seed)
    rand_key = rng.permutation(len(groups))  # seeded tie-breaking key per group
    n_splits = r.size
    demand_size = r * size.sum()
    demand_cls = r[:, None] * G.sum(axis=0)[None, :]
    assign = np.full(len(groups), -1, dtype=np.int64)

    def pick(primary: Optional[np.ndarray]) -> int:
        """Chooses a subset by remaining demand with seeded tie-breaking.

        Input:  primary — per-subset primary demand (None: screens only)
        Output: index of the chosen subset
        """
        cand = np.arange(n_splits)
        if primary is not None:
            cand = cand[np.isclose(primary[cand], primary[cand].max())]
        cand = cand[np.isclose(demand_size[cand], demand_size[cand].max())]
        return int(cand[0]) if cand.size == 1 else int(rng.choice(cand))

    def place(gi: int, s: int) -> None:
        """Assigns group gi to subset s and updates the demands.

        Input:  gi — group index; s — subset index
        Output: none
        """
        assign[gi] = s
        demand_size[s] -= size[gi]
        demand_cls[s] -= G[gi]

    if stratify and classes:
        while True:
            left = assign < 0
            remaining = G[left].sum(axis=0)
            if not (remaining > 0).any():
                break
            c = int(np.flatnonzero(remaining > 0)[np.argmin(remaining[remaining > 0])])
            todo = np.flatnonzero(left & (G[:, c] > 0))
            todo = todo[np.lexsort((rand_key[todo], -G[todo, c]))]
            for gi in todo:
                place(int(gi), pick(demand_cls[:, c]))
    for gi in np.flatnonzero(assign < 0)[np.argsort(rand_key[assign < 0], kind="stable")]:
        place(int(gi), pick(None))

    split_of_group = {g: split_names[int(assign[gi])] for gi, g in enumerate(groups)}
    return [split_of_group[g] for g in group_ids]


def audit_split(records: Sequence[ScreenRecord],
                splits: Sequence[str],
                near_dup_max_dist: Optional[int] = DEFAULT_NEAR_DUP_MAX_DIST,
                builder_prefix: bool = False,
                group_ids: Optional[Sequence[int]] = None) -> Dict[str, object]:
    """Checks a split for leakage between its subsets.

    The checks are computed directly from the records (not from group ids):
    no package (case-insensitive), builder family (if enabled), MD5 or
    near-duplicate pair (dHash distance <= k) may occur in two different
    subsets; if group_ids are given, no group may span two subsets either.

    Input:  records — screen records; splits — subset name per record;
            near_dup_max_dist — dHash radius k (None disables the check);
            builder_prefix — also check builder families;
            group_ids — optional group ids to check
    Output: report dict {'ok': True, 'n_records', 'checks': {...: 0}};
            raises SplitLeakageError listing the violations otherwise
    """
    if len(splits) != len(records):
        raise ValueError("splits must align with records")
    violations: Dict[str, List[str]] = defaultdict(list)

    def check_key(name: str, key_fn) -> None:
        """Records keys that occur in more than one subset.

        Input:  name — check name; key_fn — record -> key (None: skip)
        Output: none (appends to violations[name])
        """
        where: Dict[object, set] = defaultdict(set)
        for rec, s in zip(records, splits):
            key = key_fn(rec)
            if key is not None:
                where[key].add(s)
        for key, ss in sorted(where.items(), key=lambda t: str(t[0])):
            if len(ss) > 1:
                violations[name].append(f"{key}: {sorted(ss)}")

    check_key(REASON_PACKAGE, _package_key)
    if builder_prefix:
        check_key(REASON_BUILDER, lambda r: builder_family(r.package) if r.package else None)
    check_key(REASON_MD5, lambda r: r.md5)
    if group_ids is not None:
        if len(group_ids) != len(records):
            raise ValueError("group_ids must align with records")
        where_g: Dict[int, set] = defaultdict(set)
        for g, s in zip(group_ids, splits):
            where_g[g].add(s)
        for g, ss in sorted(where_g.items()):
            if len(ss) > 1:
                violations["group"].append(f"group {g}: {sorted(ss)}")
    if near_dup_max_dist is not None:
        # Identical hashes first (distance 0), then pairs of distinct hashes.
        splits_of: Dict[int, set] = defaultdict(set)
        stem_of: Dict[int, str] = {}
        for rec, s in zip(records, splits):
            if rec.dhash is not None:
                h = int(rec.dhash)
                splits_of[h].add(s)
                stem_of[h] = min(stem_of.get(h, rec.stem), rec.stem)
        uniq = sorted(splits_of)
        for h in uniq:
            if len(splits_of[h]) > 1:
                violations[REASON_NEAR_DUP].append(
                    f"dhash {h:016x} ({stem_of[h]}...): {sorted(splits_of[h])}")
        if len(uniq) > 1:
            for a, b, d in near_duplicate_pairs(uniq, near_dup_max_dist, _hash_bits(records)):
                ss = splits_of[uniq[a]] | splits_of[uniq[b]]
                if len(ss) > 1:
                    violations[REASON_NEAR_DUP].append(
                        f"{stem_of[uniq[a]]} ~ {stem_of[uniq[b]]}, distance {d}: {sorted(ss)}")
    checks = ["package", "md5"] + (["near_dup"] if near_dup_max_dist is not None else []) \
        + (["builder_prefix"] if builder_prefix else []) \
        + (["group"] if group_ids is not None else [])
    if any(violations.values()):
        lines = [f"{name}: {len(v)} violation(s); first: {v[:5]}"
                 for name, v in violations.items() if v]
        raise SplitLeakageError("split leaks across subsets:\n  " + "\n  ".join(lines))
    return {"ok": True, "n_records": len(records), "checks": {c: 0 for c in checks}}


# ---------------------------------------------------------------------------
# Dataset scanning
# ---------------------------------------------------------------------------

def read_class_counts(label_path: Union[str, Path]) -> Dict[int, int]:
    """Counts the instances per class in a YOLO label file.

    Input:  label_path — .txt file with "cls xc yc w h" lines (missing file = empty)
    Output: dict class id -> instance count (lines with < 5 fields are ignored)
    """
    p = Path(label_path)
    counts: Counter = Counter()
    if not p.exists():
        return {}
    for line in p.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) >= 5:
            counts[int(float(parts[0]))] += 1
    return dict(counts)


def _hash_one(path: str, size: int) -> int:
    """Computes the dHash of one file (picklable helper for worker processes).

    Input:  path — image path; size — dHash grid size
    Output: dHash int
    """
    return dhash(path, size)


def scan_yolo_dataset(dataset_dir: Union[str, Path],
                      splits: Sequence[str] = SPLITS,
                      compute_hashes: bool = True,
                      dhash_size: int = DEFAULT_DHASH_SIZE,
                      workers: int = 1) -> List[ScreenRecord]:
    """Reads a YOLO dataset directory into screen records.

    Expects images/<split>/<stem>.<ext> and labels/<split>/<stem>.txt.
    The dHash is computed once per distinct MD5 (oversampled copies are
    byte-identical to their originals and reuse the original's hash).

    Input:  dataset_dir — dataset root; splits — subset names to read;
            compute_hashes — compute MD5 and dHash; dhash_size — dHash grid;
            workers — processes for dHash computation (1 = in-process)
    Output: list of ScreenRecord sorted by (split order, stem); raises
            ValueError for a file name that parse_stem() cannot parse
    """
    root = Path(dataset_dir)
    records: List[ScreenRecord] = []
    for split in splits:
        img_dir = root / "images" / split
        if not img_dir.is_dir():
            continue
        files = sorted(p for p in img_dir.iterdir()
                       if p.is_file() and p.suffix.lower() in IMG_EXTS)
        for p in files:
            parsed = parse_stem(p.stem)
            records.append(ScreenRecord(
                stem=p.stem, path=str(p), split=split, package=parsed.package,
                class_counts=read_class_counts(root / "labels" / split / f"{p.stem}.txt"),
                is_copy=bool(parsed.copy_indices)))
    if compute_hashes:
        hash_records(records, dhash_size, workers)
    return records


def hash_records(records: Sequence[ScreenRecord], dhash_size: int = DEFAULT_DHASH_SIZE,
                 workers: int = 1) -> None:
    """Fills in md5 and dhash of records that have a path.

    The dHash is computed once per distinct MD5 (byte-identical files, such as
    oversampled copies, share it).

    Input:  records — screen records with path set; dhash_size — dHash grid;
            workers — processes for dHash computation (1 = in-process)
    Output: none (records updated in place)
    """
    for rec in records:
        rec.md5 = md5_file(rec.path)
    first_path: Dict[str, str] = {}
    for rec in records:
        first_path.setdefault(rec.md5, rec.path)
    todo = sorted(first_path.items())
    if workers > 1 and len(todo) > 1:
        with ProcessPoolExecutor(max_workers=workers) as ex:
            values = list(ex.map(_hash_one, [p for _, p in todo],
                                 [dhash_size] * len(todo), chunksize=32))
    else:
        values = [_hash_one(p, dhash_size) for _, p in todo]
    by_md5 = {m: v for (m, _), v in zip(todo, values)}
    for rec in records:
        rec.dhash = by_md5[rec.md5]


def load_class_names(yaml_path: Optional[Union[str, Path]]) -> Dict[int, str]:
    """Reads class names from a YOLO dataset.yaml ('names' list or dict).

    Input:  yaml_path — path or None
    Output: dict class id -> name (empty if the file is absent)
    """
    if yaml_path is None or not Path(yaml_path).exists():
        return {}
    import yaml  # PyYAML is a package dependency; imported lazily

    data = yaml.safe_load(Path(yaml_path).read_text(encoding="utf-8")) or {}
    names = data.get("names", [])
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    return {i: str(n) for i, n in enumerate(names)}


# ---------------------------------------------------------------------------
# Reporting
# ---------------------------------------------------------------------------

def _class_support(records: Iterable[ScreenRecord],
                   names: Mapping[int, str]) -> Dict[str, Dict[str, int]]:
    """Sums instance and image counts per class.

    Input:  records — screen records; names — class id -> name
    Output: {class name: {'instances': n, 'images': m}} for every class in
            names plus any class id found in the labels
    """
    inst: Counter = Counter()
    imgs: Counter = Counter()
    for rec in records:
        for c, n in rec.class_counts.items():
            inst[c] += n
            imgs[c] += 1
    ids = sorted(set(names) | set(inst))
    return {names.get(c, str(c)): {"instances": int(inst.get(c, 0)),
                                   "images": int(imgs.get(c, 0))} for c in ids}


def _split_stats(records: Sequence[ScreenRecord], gids: Sequence[int]) -> Dict[str, int]:
    """Counts screens, original screens, copies, apps and groups.

    Input:  records — screen records of one subset; gids — their group ids
    Output: dict of counts
    """
    return {
        "screens": len(records),
        "original_screens": sum(1 for r in records if not r.is_copy),
        "oversampled_copies": sum(1 for r in records if r.is_copy),
        "apps": len({_package_key(r) for r in records}),
        "groups": len(set(gids)),
    }


def write_manifest(path: Union[str, Path], rows: Sequence[Dict[str, object]],
                   columns: Sequence[str]) -> None:
    """Writes a CSV manifest.

    Input:  path — output .csv; rows — dicts; columns — column order
    Output: none (file written)
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(columns), extrasaction="ignore")
        w.writeheader()
        for row in rows:
            w.writerow(row)


MANIFEST_COLUMNS: Tuple[str, ...] = (
    "stem", "package", "group_id", "split", "md5", "dhash", "seen_in_train",
    "exclusion_reason", "matched_train_stem", "match_distance",
    "is_oversampled_copy", "n_instances")


def direct_matches_to_train(train: Sequence[ScreenRecord], evals: Sequence[ScreenRecord],
                            near_dup_max_dist: Optional[int] = DEFAULT_NEAR_DUP_MAX_DIST
                            ) -> Dict[str, Dict[str, int]]:
    """Counts evaluation screens that directly match a training image, by match type.

    Independent of packages and of the exclusion order, so that clone counts
    can be compared with other audits: 'md5' (byte-identical file),
    'dhash_0' (identical dHash, e.g. the same pixels re-encoded),
    'dhash_le_k' (dHash distance <= near_dup_max_dist), and the same three
    restricted to matches whose training image has a DIFFERENT package
    ('*_other_package', i.e. clones published under another name).

    Input:  train, evals — screen records (evals carry split 'val'/'test');
            near_dup_max_dist — dHash radius k (None: only md5 / dhash_0)
    Output: {split: {'md5', 'dhash_0', 'dhash_le_k', 'md5_other_package',
             'dhash_0_other_package', 'dhash_le_k_other_package'}}
    """
    pk_by_md5: Dict[str, set] = defaultdict(set)
    pk_by_hash: Dict[int, set] = defaultdict(set)
    for r in train:
        if r.md5 is not None:
            pk_by_md5[r.md5].add(_package_key(r))
        if r.dhash is not None:
            pk_by_hash[int(r.dhash)].add(_package_key(r))
    hashes = sorted(pk_by_hash)
    k = near_dup_max_dist if near_dup_max_dist is not None else 0
    index = HammingIndex(hashes, k, _hash_bits(train)) if hashes else None
    out: Dict[str, Dict[str, int]] = {}
    for split in sorted({r.split for r in evals}):
        c: Counter = Counter()
        for r in (e for e in evals if e.split == split):
            pk = _package_key(r)
            if r.md5 is not None and r.md5 in pk_by_md5:
                c["md5"] += 1
                c["md5_other_package"] += int(bool(pk_by_md5[r.md5] - {pk}))
            hits = index.query(r.dhash) if (index is not None and r.dhash is not None) else []
            exact = [i for i, d in hits if d == 0]
            if exact:
                c["dhash_0"] += 1
                c["dhash_0_other_package"] += int(any(pk_by_hash[hashes[i]] - {pk} for i in exact))
            if hits and near_dup_max_dist is not None:
                c["dhash_le_k"] += 1
                c["dhash_le_k_other_package"] += int(any(pk_by_hash[hashes[i]] - {pk} for i, _ in hits))
        out[split] = {key: int(c.get(key, 0)) for key in (
            "md5", "dhash_0", "dhash_le_k", "md5_other_package", "dhash_0_other_package",
            "dhash_le_k_other_package")}
    return out


def _stems_sha256(stems: Iterable[str]) -> str:
    """SHA-256 of a sorted stem list ("\n"-joined, UTF-8, with a final newline).

    Input:  stems — screen stems
    Output: hex digest (that of the empty string for no stems)
    """
    ordered = sorted(stems)
    text = "".join(f"{x}\n" for x in ordered)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def split_fingerprints(records: Sequence[ScreenRecord]) -> Dict[str, Dict[str, object]]:
    """Fingerprints of the split: per split the number of screens and a SHA-256 of the sorted stems.

    'originals_sha256' leaves out oversampled copies ('_os<k>' stems), so that
    the fingerprint of a training split does not depend on whether the dataset
    directory was oversampled.

    Input:  records — screen records with split set
    Output: {split: {'screens', 'originals', 'stems_sha256', 'originals_sha256'}}
    """
    out: Dict[str, Dict[str, object]] = {}
    for split in SPLITS:
        recs = [r for r in records if r.split == split]
        originals = [r.stem for r in recs if not r.is_copy]
        out[split] = {"screens": len(recs), "originals": len(originals),
                      "stems_sha256": _stems_sha256(r.stem for r in recs),
                      "originals_sha256": _stems_sha256(originals)}
    return out


def compare_fingerprints(got: Mapping[str, Mapping[str, object]],
                         expected: Mapping[str, Mapping[str, object]]) -> List[str]:
    """Compares split fingerprints with expected ones (e.g. a published summary.json).

    Only the original screens are compared ('originals', 'originals_sha256'),
    so oversampling does not matter.

    Input:  got — split_fingerprints() of this dataset; expected — the
            'split_fingerprints' entry of a reference summary.json
    Output: list of mismatch descriptions (empty if every split agrees)
    """
    problems = []
    for split in SPLITS:
        if split not in expected:
            continue
        for key in ("originals", "originals_sha256"):
            if got.get(split, {}).get(key) != expected[split].get(key):
                problems.append(f"{split}.{key}: got {got.get(split, {}).get(key)!r}, "
                                f"expected {expected[split].get(key)!r}")
    return problems


def build_unseen_report(records: Sequence[ScreenRecord],
                        names: Mapping[int, str],
                        near_dup_max_dist: Optional[int] = DEFAULT_NEAR_DUP_MAX_DIST,
                        builder_prefix: bool = False
                        ) -> Tuple[List[Dict[str, object]], Dict[str, List[ScreenRecord]],
                                   Dict[str, object]]:
    """Computes manifest rows, unseen val/test subsets and the summary.

    Input:  records — all screen records (split set); names — class names;
            near_dup_max_dist — dHash radius k; builder_prefix — flag
    Output: (manifest rows, {'val': [...], 'test': [...]} unseen records, summary dict)
    """
    train = [r for r in records if r.split == "train"]
    evals = [r for r in records if r.split in ("val", "test")]
    exclusions, gids = classify_exclusions(train, evals, near_dup_max_dist, builder_prefix)
    gid_of = {id(r): g for r, g in zip(train + evals, gids)}
    excl_of = {id(r): e for r, e in zip(evals, exclusions)}

    rows: List[Dict[str, object]] = []
    for rec in records:
        ex = excl_of.get(id(rec))
        seen = rec.split == "train" or ex is not None
        rows.append({
            "stem": rec.stem, "package": rec.package, "group_id": gid_of[id(rec)],
            "split": rec.split, "md5": rec.md5,
            "dhash": f"{rec.dhash:016x}" if rec.dhash is not None else "",
            "seen_in_train": int(seen),
            "exclusion_reason": ex.reason if ex else "",
            "matched_train_stem": ex.matched_stem if ex and ex.matched_stem else "",
            "match_distance": ex.distance if ex and ex.distance is not None else "",
            "is_oversampled_copy": int(rec.is_copy),
            "n_instances": sum(rec.class_counts.values()),
        })

    unseen: Dict[str, List[ScreenRecord]] = {}
    summary_unseen: Dict[str, object] = {}
    for split in ("val", "test"):
        full = [r for r in evals if r.split == split]
        sub = [r for r in full if excl_of[id(r)] is None]
        unseen[split] = sub
        reasons = Counter(excl_of[id(r)].reason for r in full if excl_of[id(r)] is not None)
        summary_unseen[split] = {
            **_split_stats(sub, [gid_of[id(r)] for r in sub]),
            "excluded": len(full) - len(sub),
            "excluded_by_reason": {k: reasons.get(k, 0) for k in EXCLUSION_REASONS},
        }

    train_md5 = {r.md5 for r in train if r.md5}
    group_sizes = Counter(gids)
    largest = []
    for g, n in group_sizes.most_common(10):
        recs = [r for r, gg in zip(train + evals, gids) if gg == g]
        largest.append({"group_id": g, "screens": n,
                        "packages": len({_package_key(r) for r in recs}),
                        "splits": dict(Counter(r.split for r in recs))})
    links = find_links(train + evals, near_dup_max_dist, builder_prefix)
    summary: Dict[str, object] = {
        "parameters": {"near_dup_max_dist": near_dup_max_dist,
                       "builder_prefix": builder_prefix,
                       "dhash_size": DEFAULT_DHASH_SIZE},
        "splits": {s: _split_stats([r for r in records if r.split == s],
                                   [gid_of[id(r)] for r in records if r.split == s])
                   for s in SPLITS},
        "unseen": summary_unseen,
        "byte_identical_to_train": {
            s: sum(1 for r in evals if r.split == s and r.md5 in train_md5) for s in ("val", "test")},
        "direct_matches_to_train": direct_matches_to_train(train, evals, near_dup_max_dist),
        "class_support": {
            "val_full": _class_support([r for r in evals if r.split == "val"], names),
            "val_unseen": _class_support(unseen["val"], names),
            "test_full": _class_support([r for r in evals if r.split == "test"], names),
            "test_unseen": _class_support(unseen["test"], names),
        },
        "links": {k: len(v) for k, v in links.items()},
        "largest_groups": largest,
        "split_fingerprints": split_fingerprints(records),
    }
    return rows, unseen, summary


def _write_list(path: Path, records: Sequence[ScreenRecord]) -> None:
    """Writes absolute image paths, one per line (an Ultralytics image list).

    Input:  path — output .txt; records — records with path set
    Output: none (file written)
    """
    path.write_text("".join(f"{Path(r.path).resolve()}\n" for r in records), encoding="utf-8")


def resplit_report(records: Sequence[ScreenRecord], names: Mapping[int, str],
                   ratios: Sequence[float], seed: int,
                   near_dup_max_dist: Optional[int], builder_prefix: bool
                   ) -> Tuple[List[Dict[str, object]], Dict[str, object]]:
    """Makes and audits a new group-level split of the original screens.

    Oversampled copies are left out (oversampling must be re-run on the new
    training subset).

    Input:  records — all records; names — class names; ratios, seed —
            split parameters; near_dup_max_dist, builder_prefix — grouping
    Output: (manifest rows with stem/package/group_id/split/old_split, summary dict)
    """
    originals = [r for r in records if not r.is_copy]
    gids = group_screens(originals, near_dup_max_dist, builder_prefix)
    new = split_by_group(originals, ratios, seed, True, gids)
    audit = audit_split(originals, new, near_dup_max_dist, builder_prefix, gids)
    rows = [{"stem": r.stem, "package": r.package, "group_id": g, "split": s,
             "old_split": r.split, "md5": r.md5,
             "dhash": f"{r.dhash:016x}" if r.dhash is not None else ""}
            for r, g, s in zip(originals, gids, new)]
    by_split = {s: [r for r, ss in zip(originals, new) if ss == s] for s in SPLITS}
    summary = {
        "parameters": {"ratios": list(ratios), "seed": seed,
                       "near_dup_max_dist": near_dup_max_dist,
                       "builder_prefix": builder_prefix, "stratify": True},
        "splits": {s: _split_stats(by_split[s], [g for g, ss in zip(gids, new) if ss == s])
                   for s in SPLITS},
        "class_support": {s: _class_support(by_split[s], names) for s in SPLITS},
        "audit": audit,
    }
    return rows, summary


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    """Defines and reads the command-line arguments.

    Input:  argv — argument list (default sys.argv[1:])
    Output: argparse namespace
    """
    ap = argparse.ArgumentParser(
        description="App-level leak audit, unseen-app subset and app-level split "
                    "of a YOLO-format ReDraw dataset")
    ap.add_argument("--dataset-dir", required=True,
                    help="YOLO dataset root (images/{train,val,test}, labels/{...})")
    ap.add_argument("--out", default="results/splits", help="output directory")
    ap.add_argument("--dataset-yaml", default=None,
                    help="dataset.yaml with class names (default: <dataset-dir>/dataset.yaml)")
    ap.add_argument("--near-dup-dist", type=int, default=DEFAULT_NEAR_DUP_MAX_DIST,
                    help="max dHash Hamming distance for near-duplicates (provisional default 4; "
                         "negative disables near-duplicate grouping)")
    ap.add_argument("--builder-prefix", action="store_true",
                    help="also group apps of the same builder family (e.g. com.andromo.devNNNN)")
    ap.add_argument("--resplit", action="store_true",
                    help="also write a new app-level split manifest (for retraining), audited")
    ap.add_argument("--ratios", type=float, nargs=3, default=list(DEFAULT_RATIOS),
                    help="train/val/test ratios for --resplit")
    ap.add_argument("--seed", type=int, default=DEFAULT_SEED, help="seed for --resplit")
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 1),
                    help="processes for dHash computation")
    ap.add_argument("--expect-fingerprints", default=None, metavar="SUMMARY_JSON",
                    help="reference summary.json: exit with status 3 if the original screens "
                         "of any split differ from its split_fingerprints")
    return ap.parse_args(argv)


def main(argv: Optional[Sequence[str]] = None) -> int:
    """Runs the leak audit and writes the manifest, unseen lists and summary.

    Input:  argv — argument list (default sys.argv[1:])
    Output: exit code 0 (1 without images, 3 if --expect-fingerprints does not
            match); files written to --out: manifest.csv, unseen_val.txt,
            unseen_test.txt, summary.json and, with --resplit,
            app_split_manifest.csv and app_split_summary.json
    """
    args = parse_args(argv)
    dataset_dir = Path(args.dataset_dir)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    k = args.near_dup_dist if args.near_dup_dist >= 0 else None
    yaml_path = Path(args.dataset_yaml) if args.dataset_yaml else dataset_dir / "dataset.yaml"
    names = load_class_names(yaml_path)

    records = scan_yolo_dataset(dataset_dir, workers=args.workers)
    if not records:
        print(f"[ERR] no images found under {dataset_dir}/images/{{train,val,test}}")
        return 1
    rows, unseen, summary = build_unseen_report(records, names, k, args.builder_prefix)
    summary["dataset_dir"] = str(dataset_dir)
    write_manifest(out / "manifest.csv", rows, MANIFEST_COLUMNS)
    _write_list(out / "unseen_val.txt", unseen["val"])
    _write_list(out / "unseen_test.txt", unseen["test"])
    summary["note"] = ("unseen_*.txt contain absolute paths (machine-specific); "
                       "manifest.csv is the portable record of the subset")
    (out / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    for split in ("val", "test"):
        u = summary["unseen"][split]
        print(f"{split}: {summary['splits'][split]['screens']} screens -> "
              f"{u['screens']} unseen ({u['apps']} apps); excluded {u['excluded']} "
              f"{u['excluded_by_reason']}")

    if args.resplit:
        rs_rows, rs_summary = resplit_report(records, names, args.ratios, args.seed,
                                             k, args.builder_prefix)
        write_manifest(out / "app_split_manifest.csv", rs_rows,
                       ("stem", "package", "group_id", "split", "old_split", "md5", "dhash"))
        (out / "app_split_summary.json").write_text(json.dumps(rs_summary, indent=2),
                                                    encoding="utf-8")
        print(f"app-level split: { {s: v['screens'] for s, v in rs_summary['splits'].items()} } "
              f"audit ok")
    for split, fp in summary["split_fingerprints"].items():
        print(f"{split}: {fp['originals']} original screens, sha256 {fp['originals_sha256'][:16]}...")
    print(f"Outputs: {out.resolve()}")
    if args.expect_fingerprints:
        ref = json.loads(Path(args.expect_fingerprints).read_text(encoding="utf-8"))
        problems = compare_fingerprints(summary["split_fingerprints"],
                                        ref.get("split_fingerprints", {}))
        if problems or "split_fingerprints" not in ref:
            print("[FAIL] the split differs from " + args.expect_fingerprints + ": "
                  + ("; ".join(problems) or "no split_fingerprints in the reference"))
            return 3
        print(f"[OK] split fingerprints equal {args.expect_fingerprints}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
#@title Step 7b - Verify the embedded app_split.py, run it, cross-check with the thesis
from collections import defaultdict
src = Path(PINNED['app_split_file'])
got = file_hash(src)
if got != PINNED['app_split_sha256']:
    fixed = src.read_text(encoding='utf-8').rstrip('\n') + '\n'      # only a trailing-newline difference is tolerated
    if hashlib.sha256(fixed.encode('utf-8')).hexdigest() == PINNED['app_split_sha256']:
        src.write_text(fixed, encoding='utf-8')
        got = file_hash(src)
    else:
        raise PipelineStop(f'{src} has SHA-256 {got}, expected {PINNED["app_split_sha256"]}: the embedded tool was '
                           'modified. Re-upload the original notebook.')
tool = D['tools'] / 'app_split.py'
shutil.copyfile(src, tool)
print(f'app_split.py OK (sha256 {got}) -> {tool}')

# The second run (--near-dup-dist -1) executes the tool's own main() unchanged, but takes MD5 + dHash of every
# screen from the manifest of the first run instead of decoding all 18,694 PNGs again (saves ~12-15 min).
_CACHED_HASHES = r'''
import csv, importlib.util, sys
tool, manifest = sys.argv[1], sys.argv[2]
spec = importlib.util.spec_from_file_location("app_split", tool)
m = importlib.util.module_from_spec(spec)
sys.modules["app_split"] = m
spec.loader.exec_module(m)
with open(manifest, newline="", encoding="utf-8") as f:
    known = {r["stem"]: (r["md5"], int(r["dhash"], 16)) for r in csv.DictReader(f) if r["md5"] and r["dhash"]}
_orig = m.hash_records
def hash_records(records, dhash_size=m.DEFAULT_DHASH_SIZE, workers=1):
    if dhash_size == m.DEFAULT_DHASH_SIZE and records and all(r.stem in known for r in records):
        for r in records:
            r.md5, r.dhash = known[r.stem]
        print(f"[cached] MD5 + dHash of {len(records)} screens taken from {manifest}", flush=True)
    else:
        _orig(records, dhash_size, workers)
m.hash_records = hash_records
sys.exit(m.main(sys.argv[3:]))
'''

act = load_state('active')
yolo, yaml_path = Path(act['yolo']), Path(act['yaml'])
out_full, out_pm = D['splits'] / act['id'] / 'full', D['splits'] / act['id'] / 'package_md5'
workers = max(1, os.cpu_count() or 1)
fp_status = None
for key, out, extra in ((f'appsplit_{act["id"]}_full', out_full, []),
                        (f'appsplit_{act["id"]}_package_md5', out_pm, ['--near-dup-dist', '-1'])):
    if done(key):
        print(f'[skip] {key} already done')
        fp_status = marker(key).get('fingerprints', fp_status)
        continue
    args = ['--dataset-dir', yolo, '--out', out, '--dataset-yaml', yaml_path, '--workers', workers] + extra
    if EXPECT_FINGERPRINTS_JSON and not extra:
        args += ['--expect-fingerprints', EXPECT_FINGERPRINTS_JSON]
    if extra:
        print('\napp_split, package + MD5 grouping (hashes reused from the first run) ...', flush=True)
        cmd = [sys.executable, '-I', '-c', _CACHED_HASHES, tool, out_full / 'manifest.csv'] + args
    else:
        print(f'\napp_split, full grouping: hashing every screen ({workers} processes; ~12-16 min on Colab) ...',
              flush=True)
        cmd = [sys.executable, '-I', tool] + args
    rc, _ = run(cmd, log='app_split.log', check=False)
    if rc not in (0, 3):
        raise PipelineStop(f'app_split failed with exit code {rc} (log: {D["logs"] / "app_split.log"})')
    info = {'command': qcmd(cmd if not extra else ['python', '-I', tool] + args) + (
        '   # run through main() with MD5/dHash read from the full-grouping manifest' if extra else ''),
        'exit_code': rc}
    if EXPECT_FINGERPRINTS_JSON and not extra:
        info['fingerprints'] = fp_status = 'match' if rc == 0 else 'MISMATCH'
    mark(key, info)

summary = jload(out_full / 'summary.json')
summary_pm = jload(out_pm / 'summary.json')
with open(out_full / 'manifest.csv', newline='', encoding='utf-8') as f:
    manifest = list(csv.DictReader(f))
train_pk = {r['package'].lower() for r in manifest if r['split'] == 'train'}
pkg_only = {}
for s in ('val', 'test'):
    rows_s = [r for r in manifest if r['split'] == s]
    apps = {r['package'].lower() for r in rows_s}
    unseen = sorted(r['stem'] for r in rows_s if r['package'].lower() not in train_pk)
    pkg_only[s] = {'screens': len(rows_s), 'unseen_screens': len(unseen), 'apps': len(apps),
                   'apps_also_in_train': len(apps & train_pk), 'unseen_stems': unseen}
    (out_full.parent / f'unseen_{s}_package_only_stems.txt').write_text(''.join(x + '\n' for x in unseen), encoding='utf-8')
for s in SPLITS:  # the split itself, archived as stem lists
    (out_full.parent / f'split_{s}_stems.txt').write_text(
        ''.join(r['stem'] + '\n' for r in sorted(manifest, key=lambda r: r['stem']) if r['split'] == s), encoding='utf-8')

# "Pixel-identical to a training image" (Table A.2: 9 val, 8 test): candidates are training screens with the
# same dHash; equal MD5 means equal bytes, otherwise the decoded pixels are compared.
pkey = f'pixel_identical_{act["id"]}'
if done(pkey):
    pix = marker(pkey)['result']
else:
    from PIL import Image
    path_of = {(s, p.stem): p for s in SPLITS for p in list_images(yolo, s)}
    train_by_hash = defaultdict(list)
    for r in manifest:
        if r['split'] == 'train':
            train_by_hash[r['dhash']].append(r)

    def _pixels(p):
        with Image.open(p) as im:
            return im.size, im.convert('RGBA').tobytes()

    pix, pairs, t0 = {}, [], time.time()
    for s in ('val', 'test'):
        c = Counter()
        for r in (x for x in manifest if x['split'] == s):
            pk, cands = r['package'].lower(), train_by_hash.get(r['dhash'], [])
            if not cands:
                continue
            c['dhash_identical'] += 1
            # other packages first, within them equal bytes first (no decoding); at most 30 decodes per screen
            cands = sorted(cands, key=lambda t: (t['package'].lower() == pk, t['md5'] != r['md5'], t['stem']))
            mine, any_hit, other_hit, decodes = None, None, None, 0
            for t in cands:
                same_pkg = t['package'].lower() == pk
                if other_hit or (any_hit and same_pkg):
                    break   # both answers known (other-package candidates are all checked before same-package ones)
                if t['md5'] == r['md5']:
                    hit = True
                elif decodes < 30:
                    mine = mine or _pixels(path_of[(s, r['stem'])])
                    decodes += 1
                    hit = _pixels(path_of[('train', t['stem'])]) == mine
                else:
                    c['decode_limit_reached'] += 1
                    break
                if hit:
                    any_hit = any_hit or t
                    if not same_pkg:
                        other_hit = t
            if any_hit:
                best = other_hit or any_hit
                c['pixels'] += 1
                c['pixels_other_package'] += other_hit is not None
                pairs.append({'split': s, 'stem': r['stem'], 'train_stem': best['stem'],
                              'other_package': int(other_hit is not None),
                              'byte_identical': int(best['md5'] == r['md5'])})
        pix[s] = {k: int(c.get(k, 0)) for k in ('dhash_identical', 'pixels', 'pixels_other_package',
                                               'decode_limit_reached')}
    with open(out_full.parent / 'pixel_identical_to_train.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['split', 'stem', 'train_stem', 'other_package', 'byte_identical'])
        w.writeheader()
        w.writerows(pairs)
    mark(pkey, {'result': pix, 'seconds': round(time.time() - t0)})

E = EXPECTED
dm = summary.get('direct_matches_to_train', {})
rows = [check_row('test screens unseen by package (from the manifest; = Step 5)', E['unseen_test_package_only'],
                  pkg_only['test']['unseen_screens']),
        check_row('test apps (distinct packages)', E['test_apps'], pkg_only['test']['apps']),
        check_row('test apps that also occur in train', E['test_apps_in_train'], pkg_only['test']['apps_also_in_train'])]
for s in ('val', 'test'):
    exp, p = E[f'{s}_pixel_identical_to_train'], pix[s]
    rows.append(check_row(f'{s} screens pixel-identical to a train screen (Table A.2)', exp, p['pixels'],
                          status='PASS' if p['pixels'] == exp else 'WARN',
                          note=f"other package {p['pixels_other_package']}; byte-identical (MD5) "
                               f"{summary['byte_identical_to_train'][s]} (other package "
                               f"{dm.get(s, {}).get('md5_other_package')}); identical dHash {p['dhash_identical']}"
                               + (f"; decode limit reached for {p['decode_limit_reached']}" if p['decode_limit_reached'] else '')))
rows += [check_row('val screens unseen by package', 'reported', pkg_only['val']['unseen_screens'], status='INFO'),
         check_row('test unseen, package + MD5', 'reported', summary_pm['unseen']['test']['screens'], status='INFO'),
         check_row('test unseen, full grouping (package+MD5+dHash<=4)', 'reported',
                   summary['unseen']['test']['screens'], status='INFO',
                   note=f"excluded by reason {summary['unseen']['test']['excluded_by_reason']}"),
         check_row('val unseen, full grouping (package+MD5+dHash<=4)', 'reported',
                   summary['unseen']['val']['screens'], status='INFO'),
         check_row('manifest rows = screens', sum(act['counts'][s]['images'] for s in SPLITS), len(manifest))]
for s in SPLITS:
    fp = summary['split_fingerprints'][s]
    rows.append(check_row(f'{s} fingerprint app_split = Step 5', act['counts'][s]['stems_sha256'][:16] + '...',
                          fp['originals_sha256'][:16] + '...', ok=fp['originals_sha256'] == act['counts'][s]['stems_sha256']))
if fp_status:
    rows.append(check_row('split fingerprints = EXPECT_FINGERPRINTS_JSON', 'match', fp_status))
set_checks('7 manifest cross-checks', rows)
headline = {'unseen_test_package_only': pkg_only['test']['unseen_screens'],
            'unseen_test_package_md5': summary_pm['unseen']['test']['screens'],
            'unseen_test_full': summary['unseen']['test']['screens'],
            'unseen_val_package_only': pkg_only['val']['unseen_screens'],
            'unseen_val_package_md5': summary_pm['unseen']['val']['screens'],
            'unseen_val_full': summary['unseen']['val']['screens'],
            'test_apps': pkg_only['test']['apps'], 'test_apps_in_train': pkg_only['test']['apps_also_in_train'],
            'pixel_identical_to_train': {s: pix[s]['pixels'] for s in ('val', 'test')},
            'pixel_identical_other_package': {s: pix[s]['pixels_other_package'] for s in ('val', 'test')},
            'md5_identical_to_train': summary['byte_identical_to_train'],
            'split_fingerprints': {s: summary['split_fingerprints'][s]['originals_sha256'] for s in SPLITS}}
save_state('appsplit', {'dataset': act['id'], 'tool_sha256': got, 'out_full': str(out_full), 'out_package_md5': str(out_pm),
                        'package_only': {s: {k: v for k, v in d.items() if k != 'unseen_stems'} for s, d in pkg_only.items()},
                        'pixel_identical': pix, 'headline': headline})
print('\n' + json.dumps(headline, indent=2))
write_diagnostics()

## Step 8 – Build the zip for the laptop and copy it to Google Drive

This step runs only if Step 5 PASSed or `FORCE` is ticked. Otherwise it refreshes `REPORT.txt` and `ui2compose_diagnostics.zip` on Drive and stops. Content of `ui2compose_redraw_valtest_<dataset>.zip` (about 2.2–2.6 GB; PNGs stored uncompressed, text compressed):

| path in the zip | content |
|---|---|
| `dataset/images/{val,test}/` | val + test screenshots, named `<package>-screens_screenshot_<n>.png` |
| `dataset/labels/{train,val,test}/` | YOLO labels of **all** splits |
| `dataset/dataset.yaml` | portable dataset config (the folder of the yaml is the dataset root) |
| `dataset/unseen_{val,test}.txt`, `..._package_md5.txt`, `..._package_only.txt` | unseen-app image lists (`./images/<split>/<file>`): full grouping, package + MD5, package only |
| `raw_xml/{val,test}/<package>-screens/hierarchy_<n>.xml` | raw ReDraw view hierarchies of the val + test screens |
| `raw_xml/mapping.csv` | YOLO image ↔ raw app folder / hierarchy / screenshot, with MD5/SHA-256 |
| `splits/` | `manifest.csv` (MD5 + dHash of **all** 18,694 screens), `summary.json` (with `split_fingerprints`), stem lists of every split, `pixel_identical_to_train.csv` |
| `verification/` | all PASS/FAIL tables, per-class counts, metric JSONs |
| `provenance/` | tags + commits, pinned hashes, Zenodo MD5, settings, environment, `pip freeze`, logs, the `app_split.py` that ran |

The unseen lists are rebuilt from `manifest.csv` (app_split's own `unseen_*.txt` contain absolute Colab paths), and the step checks that every listed image is in the zip and that each list has the size `app_split` reported. The zip is checked (CRC of every member), copied to `MyDrive/ui2compose_revision/` after a check of the free Drive space, read back and compared by SHA-256. The final `REPORT.txt` goes next to it. If the copy fails (Drive full or disconnected), the zip stays on the Colab machine and running this cell again only repeats the copy.

In [ ]:
#@title Step 8 - Build the zip, copy it to Google Drive, write REPORT.txt
act = load_state('active')
aps = load_state('appsplit')
if not act or not aps:
    raise PipelineStop('Steps 5-7 have not finished - run them first')
if not act['split_ok'] and not FORCE:
    write_diagnostics()
    text, dest = write_report()
    drive_flush()
    raise PipelineStop('the thesis split was NOT reproduced (Step 5 FAIL), so no zip was built. '
                       f'REPORT.txt and {DIAG_ZIP} were written to {dest.parent if dest else D["out"]} - please send '
                       'both back. To build the zip anyway: tick FORCE in Step 1 and choose Run all again.')
tag_note = '' if act['split_ok'] else '_UNVERIFIED'
zip_name = f'ui2compose_redraw_valtest_{act["id"]}{tag_note}.zip'
top = zip_name[:-4]
yolo, rr = Path(act['yolo']), raw_root()
out_full, out_pm = Path(aps['out_full']), Path(aps['out_package_md5'])
pkey = f'package_{act["id"]}{tag_note}'


def unseen_lists():
    """The unseen-app image lists, rebuilt from the manifests (app_split's own unseen_*.txt hold absolute
    paths that follow the --symlink links to the raw ReDraw files, so they are not usable on the laptop).
    Returns [(arcname, split, stems, expected count, grouping)]."""
    specs = []
    for s in ('val', 'test'):
        for grouping, d, suffix in (('full grouping', out_full, ''), ('package + MD5', out_pm, '_package_md5')):
            with open(d / 'manifest.csv', newline='', encoding='utf-8') as f:
                stems = sorted(r['stem'] for r in csv.DictReader(f) if r['split'] == s and r['seen_in_train'] == '0')
            specs.append((f'dataset/unseen_{s}{suffix}.txt', s, stems, jload(d / 'summary.json')['unseen'][s]['screens'],
                          grouping))
        pk = [l for l in (out_full.parent / f'unseen_{s}_package_only_stems.txt').read_text(encoding='utf-8').splitlines() if l]
        specs.append((f'dataset/unseen_{s}_package_only.txt', s, pk, aps['package_only'][s]['unseen_screens'],
                      'package only'))
    return specs


ext_of = {s: {p.stem: p.suffix for p in list_images(yolo, s)} for s in ('val', 'test')}
UNSEEN = unseen_lists()

if done(pkey) and Path(marker(pkey)['zip']).exists():
    pinfo = marker(pkey)
    print(f'[skip] zip already built: {pinfo["zip"]} ({pinfo["size_human"]})')
else:
    manifest = {}
    with open(out_full / 'manifest.csv', newline='', encoding='utf-8') as f:
        for r in csv.DictReader(f):
            manifest[r['stem']] = r
    names = read_names(act['yaml'])
    members = []  # (source path or None, arcname, text or None, compress)
    S, Z = zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED
    for s in ('val', 'test'):
        for p in list_images(yolo, s):
            members.append((p, f'dataset/images/{s}/{p.name}', None, S))
    for s in SPLITS:
        for p in sorted((yolo / 'labels' / s).glob('*.txt')):
            members.append((p, f'dataset/labels/{s}/{p.name}', None, Z))
    yaml_text = ('# Portable copy: no "path" key, so Ultralytics uses the folder of this file as the dataset root.\n'
                 '# images/train is not part of this package (labels/train is).\n'
                 'train: images/train\nval: images/val\ntest: images/test\n'
                 f'nc: {len(names)}\nnames:\n' + ''.join(f'  - {n}\n' for n in names))
    members.append((None, 'dataset/dataset.yaml', yaml_text, Z))
    members.append((Path(act['yaml']), 'dataset/dataset_as_built_on_colab.yaml', None, Z))
    for arc, s, stems, _, grouping in UNSEEN:   # relative paths: "./" = the folder of the list (dataset/)
        members.append((None, arc, ''.join(f'./images/{s}/{x}{ext_of[s].get(x, ".png")}\n' for x in stems), Z))
    # raw view hierarchies of val + test, with a mapping table
    mapping, problems = [], []
    for s in ('val', 'test'):
        for p in list_images(yolo, s):
            m = STEM_RE.match(p.stem)
            app, n = m.group('app'), m.group('n')
            xml = rr / app / f'hierarchy_{n}.xml'
            shot = next((rr / app / f'screenshot_{n}{e}' for e in ('.png', '.jpg', '.jpeg')
                         if (rr / app / f'screenshot_{n}{e}').exists()), None)
            img_md5 = manifest.get(p.stem, {}).get('md5') or file_hash(p, 'md5')
            shot_md5 = file_hash(shot, 'md5') if shot else None
            if not xml.exists() or shot_md5 != img_md5:
                problems.append(p.stem)
            arc_xml = f'raw_xml/{s}/{app}/hierarchy_{n}.xml'
            if xml.exists():
                members.append((xml, arc_xml, None, Z))
            mapping.append({'split': s, 'stem': p.stem, 'package': app[:-len('-screens')], 'screen_index': n,
                            'image_in_zip': f'dataset/images/{s}/{p.name}', 'label_in_zip': f'dataset/labels/{s}/{p.stem}.txt',
                            'xml_in_zip': arc_xml if xml.exists() else '',
                            'raw_app_dir': app, 'raw_hierarchy': f'{app}/hierarchy_{n}.xml',
                            'raw_screenshot': f'{app}/{shot.name}' if shot else '',
                            'image_md5': img_md5, 'raw_screenshot_md5': shot_md5 or '',
                            'xml_sha256': file_hash(xml) if xml.exists() else ''})
    buf = io.StringIO()
    w = csv.DictWriter(buf, fieldnames=list(mapping[0]) if mapping else ['split'])
    w.writeheader()
    w.writerows(mapping)
    members.append((None, 'raw_xml/mapping.csv', buf.getvalue(), Z))
    # splits (app_split's unseen_*.txt with absolute Colab paths are left out: see dataset/unseen_*.txt)
    for sub, d in (('full', out_full), ('package_md5', out_pm)):
        for p in sorted(d.iterdir()):
            if p.is_file() and not p.name.startswith('unseen_'):
                members.append((p, f'splits/{sub}/{p.name}', None, Z))
    for p in sorted(list(out_full.parent.glob('*.txt')) + list(out_full.parent.glob('*.csv'))):
        members.append((p, f'splits/{p.name}', None, Z))
    for name in ('verification', 'checks', 'active', 'metrics_summary', 'appsplit', 'examples_check', 'env', 'zenodo',
                 'primary'):
        if (D['state'] / f'{name}.json').exists():
            members.append((D['state'] / f'{name}.json', f'verification/{name}.json', None, Z))
    for p in sorted(D['metrics'].glob(f'metrics_{act["id"]}_*.json')):
        members.append((p, f'verification/{p.name}', None, Z))
    members.append((None, 'verification/per_class_counts.txt', '\n'.join(class_table(act['counts'])) + '\n', Z))
    freeze = run([sys.executable, '-m', 'pip', 'freeze'], echo=False, check=False)[1]
    members.append((None, 'provenance/pip_freeze.txt', freeze, Z))
    prov = {'written': now(), 'dry_run': DRY_RUN, 'force': FORCE, 'dataset_used': act['id'], 'split_ok': act['split_ok'],
            'repo_url': REPO_URL, 'repo_tag': REPO_TAG, 'fallback_tags': FALLBACK_TAGS,
            'clones': {t: marker(f'clone_{t}') for t in [REPO_TAG] + FALLBACK_TAGS if done(f'clone_{t}')},
            'preprocessing': {f.stem: jload(f) for f in sorted(D['markers'].glob('parse_*.done'))
                              + sorted(D['markers'].glob('build_*.done'))},
            'pinned': PINNED, 'zenodo': load_state('zenodo'), 'md5_check': marker('md5'), 'extract': marker('extract'),
            'raw_scans': {k: marker(k) for k in ('rawscan_names', 'rawscan_pairs')},
            'settings': {'USE_SYMLINKS': USE_SYMLINKS, 'VAL_SETTINGS': VAL_SETTINGS, 'MAP_TOL': MAP_TOL,
                         'CLASS_TOL': CLASS_TOL, 'REEVAL_TOL': REEVAL_TOL, 'ULTRALYTICS_VERSION': ULTRALYTICS_VERSION},
            'env': load_state('env'), 'app_split_sha256': aps['tool_sha256'],
            'raw_mapping_problems': problems}
    members.append((None, 'provenance/provenance.json', json.dumps(prov, indent=2, default=str), Z))
    members.append((D['tools'] / 'app_split.py', 'provenance/app_split.py', None, Z))
    for p in sorted(D['logs'].glob('*.log')):
        members.append((p, f'provenance/logs/{p.name}', None, Z))
    readme = f'''ui2compose - ReDraw val/test package for the SoftwareX revision
written {now()} on Colab; dataset variant: {act["id"]}; split reproduced: {act["split_ok"]}

dataset/            val + test images, labels of train/val/test, dataset.yaml (portable: no "path" key)
dataset/unseen_<split>.txt               unseen-app images, full grouping (package OR MD5 OR dHash <= 4)
dataset/unseen_<split>_package_md5.txt   unseen-app images, package OR MD5
dataset/unseen_<split>_package_only.txt  unseen-app images, package only (thesis: 214 test screens)
                    (lines are ./images/<split>/<file>, relative to dataset/, as Ultralytics expects)
raw_xml/            raw ReDraw uiautomator hierarchies of the val + test screens; mapping.csv links every
                    YOLO image to its ReDraw app folder, hierarchy_<n>.xml and screenshot_<n>.png
splits/full/        app_split (v1.2.0) on all splits: manifest.csv (md5 + dhash of every screen, seen_in_train),
                    summary.json (split_fingerprints)
splits/package_md5/ the same with --near-dup-dist -1
splits/*.txt, *.csv stem lists of the train/val/test split and of the package-only unseen subsets;
                    pixel_identical_to_train.csv (val/test screens with the same pixels as a train screen)
verification/       PASS/FAIL tables (checks.json), per-class counts, YOLO.val metrics (metrics_*.json)
provenance/         tags/commits/hashes, Zenodo MD5, settings, environment, pip freeze, logs, app_split.py

Examples (laptop, run from the unzipped folder; runs/ is the repository's runs/ folder):
  yolo val model=runs/oversample_5k/weights/best.pt data=dataset/dataset.yaml split=test
  ui2compose-evaluate-detector --model runs/oversample_5k/weights/best.pt --images dataset/unseen_test.txt \\
      --labels dataset/labels/test --dataset-yaml dataset/dataset.yaml --groups splits/full/manifest.csv \\
      --bootstrap 1000 --out results/unseen_test_metrics.json
  ui2compose-app-split --dataset-dir <regenerated data/yolo> --out /tmp/check --expect-fingerprints splits/full/summary.json
'''
    members.append((None, 'README_PACKAGE.txt', readme, Z))
    text, _ = write_report()
    members.append((None, 'REPORT.txt', text, Z))

    zip_path = D['out'] / zip_name
    tmp = zip_path.with_name(zip_path.name + '.partial')
    t0 = time.time()
    print(f'writing {len(members):,} files into {zip_path} ...', flush=True)
    with zipfile.ZipFile(tmp, 'w', allowZip64=True) as zf:
        for i, (src_p, arc, txt, comp) in enumerate(members, 1):
            if txt is not None:
                zi = zipfile.ZipInfo(f'{top}/{arc}', date_time=time.localtime()[:6])
                zi.external_attr = 0o644 << 16  # readable file permissions after unzipping
                zf.writestr(zi, txt, compress_type=comp)
            else:
                zf.write(src_p, f'{top}/{arc}', compress_type=comp)
            if i % 5000 == 0:
                print(f'  {i:,} / {len(members):,}', flush=True)
    with zipfile.ZipFile(tmp) as zf:
        bad = zf.testzip()
    if bad:
        raise PipelineStop(f'zip test failed at {bad}')
    os.replace(tmp, zip_path)
    size = zip_path.stat().st_size
    pinfo = {'zip': str(zip_path), 'name': zip_name, 'files': len(members), 'size_bytes': size, 'size_human': human(size),
             'sha256': file_hash(zip_path), 'raw_mapping_problems': len(problems), 'seconds': round(time.time() - t0)}
    mark(pkey, pinfo)
save_state('package', pinfo)


def _zip_group(name):
    parts = name.split('/')[1:]
    if parts[0] == 'dataset' and len(parts) >= 4:
        return '/'.join(parts[:3])          # dataset/images/val, dataset/labels/train, ...
    if parts[0] == 'raw_xml' and len(parts) >= 4:
        return '/'.join(parts[:2])          # raw_xml/val, raw_xml/test
    return '/'.join(parts[:-1]) or '(top level)'


counts_in_zip = Counter()
with zipfile.ZipFile(pinfo['zip']) as zf:
    namelist = set(zf.namelist())
    lists_in_zip = {a: zf.read(f'{top}/{a}').decode('utf-8').splitlines() for a, *_ in UNSEEN if f'{top}/{a}' in namelist}
for n in namelist:
    counts_in_zip[_zip_group(n)] += 1
print('zip content (files per folder):')
for k, v in sorted(counts_in_zip.items()):
    print(f'  {v:>7,}  {k}')
rows = [check_row('zip size', '< 4 GB (laptop: ~15 GB free)', pinfo['size_human'], ok=pinfo['size_bytes'] < 4e9,
                  note='OK' if pinfo['size_bytes'] < 3.5e9 else 'close to the limit'),
        check_row('val images in zip', act['counts']['val']['images'], counts_in_zip.get('dataset/images/val', 0)),
        check_row('test images in zip', act['counts']['test']['images'], counts_in_zip.get('dataset/images/test', 0)),
        check_row('label files in zip', sum(act['counts'][s]['label_files'] for s in SPLITS),
                  sum(counts_in_zip.get(f'dataset/labels/{s}', 0) for s in SPLITS)),
        check_row('raw XML of val + test in zip', act['counts']['val']['images'] + act['counts']['test']['images'],
                  counts_in_zip.get('raw_xml/val', 0) + counts_in_zip.get('raw_xml/test', 0)),
        check_row('val/test images that map to their raw ReDraw files (MD5)', 0, pinfo['raw_mapping_problems'],
                  note='number of problems')]
for arc, s, stems, expected_n, grouping in UNSEEN:   # every listed image must be in the zip
    lines = lists_in_zip.get(arc, [])
    found = sum(1 for l in lines if l.startswith('./') and f'{top}/dataset/{l[2:]}' in namelist)
    rows.append(check_row(f'{arc} ({grouping}): listed images present in the zip', expected_n, found,
                          ok=found == len(lines) == expected_n, note=f'{len(lines)} lines'))

# copy to Google Drive
dest = Path(OUT_DRIVE_DIR) / pinfo['name']
dkey = f'drive_{pinfo["sha256"][:16]}'
local_note = f'The zip is ready at {pinfo["zip"]} (nothing is rebuilt): run this cell (Step 8) again to copy it.'
if not drive_alive():
    try:
        mount_drive(force=True)
    except PipelineStop as e:
        write_report()
        raise PipelineStop(f'{e} {local_note}')
if done(dkey) and dest.exists() and dest.stat().st_size == pinfo['size_bytes']:
    print(f'[skip] already on Drive: {dest}')
    drive_ok = marker(dkey).get('sha256_ok')
else:
    tmp_d = Path(str(dest) + '.partial')
    if tmp_d.exists():
        tmp_d.unlink()   # an interrupted earlier copy
    dfree = drive_free_gb()
    if dfree is not None and dfree < pinfo['size_bytes'] / 1e9 + 0.05:
        write_report()
        raise PipelineStop(f'Google Drive has only {dfree:.2f} GB free, the zip needs {pinfo["size_human"]}. Free up '
                           f'space on drive.google.com (and empty its bin). {local_note}')
    print(f'copying {pinfo["size_human"]} to {dest} ...', flush=True)
    t0 = time.time()
    try:
        shutil.copyfile(pinfo['zip'], tmp_d)
        os.replace(tmp_d, dest)
        drive_ok = (file_hash(dest) == pinfo['sha256']) if VERIFY_DRIVE_COPY else None
        (Path(OUT_DRIVE_DIR) / (pinfo['name'] + '.sha256')).write_text(f'{pinfo["sha256"]}  {pinfo["name"]}\n')
    except OSError as e:
        raise PipelineStop(f'copying the zip to Google Drive failed ({e}). Check that the Drive is connected and '
                           f'not full. {local_note}')
    mark(dkey, {'path': str(dest), 'sha256_ok': drive_ok, 'seconds': round(time.time() - t0)})
rows.append(check_row('zip on Drive: size', pinfo['size_bytes'], dest.stat().st_size))
rows.append(check_row('zip on Drive: SHA-256 read back', True, drive_ok, status=None if VERIFY_DRIVE_COPY else 'INFO'))
pinfo.update({'drive_path': str(dest), 'drive_sha256_ok': drive_ok})
save_state('package', pinfo)
set_checks('8 package', rows)
text, rep = write_report(final=True)
banner(f'DONE. Zip on Google Drive: {dest}\n      size {pinfo["size_human"]}, sha256 {pinfo["sha256"][:16]}...\n'
       f'REPORT: {rep}')

## Step 9 – Finished: what to send back

1. **Send `REPORT.txt` back.** It is in `MyDrive/ui2compose_revision/` and printed below. Its *KEY NUMBERS* block has the numbers that matter.
2. Download `ui2compose_redraw_valtest_<dataset>.zip` and its `.sha256` file from that folder to the laptop. Google Drive warns that it "can't scan this file for viruses" because the file is large: choose **Download anyway**. On the laptop, `shasum -a 256 <zip>` must print the value in the `.sha256` file.
3. The Colab machine can then be deleted (*Runtime → Disconnect and delete runtime*).

In [ ]:
#@title Step 9 - Print REPORT.txt and flush Google Drive
text, rep = write_report(final=True)
print(text)
print('REPORT.txt:', rep)
drive_flush()   # makes sure everything is written to Drive